# Data PreProcessing #

## Cleaning up Data ##

 - Remove reviews without rating 
 - Handle invalid timestamps
 - Convert rating to float

In [8]:
import glob
import os
import pandas as pd

# ══════════════════════════════════════════════════════════════════════
# NETTOYAGE DES DONNÉES (Data Preprocessing)
#
# Ce script parcourt tous les fichiers Parquet d'échantillons et
# applique trois étapes de nettoyage :
#   1. Suppression des reviews sans note (rating manquant ou invalide)
#   2. Filtrage des timestamps invalides (hors plage temporelle)
#   3. Conversion du rating en float pour cohérence numérique
#
# À la fin, chaque fichier est sauvegardé en version nettoyée.
# ══════════════════════════════════════════════════════════════════════

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))

# Amazon a été fondé en juillet 1995 — aucune review ne peut être antérieure.
# On borne aussi par le futur pour exclure les timestamps manifestement erronés.
MIN_DATE = pd.Timestamp("1995-07-01")
MAX_DATE = pd.Timestamp("2025-12-31")

for path in SAMPLE_PATHS:
    # ── Garde : fichiers trop petits (probablement corrompus) ──────
    if os.path.getsize(path) < 1024:
        print(f"  ⚠ Fichier ignoré : {path} ({os.path.getsize(path)} octets — probablement corrompu)")
        continue

    print(f"\n{'═' * 60}")
    print(f"  Fichier : {path}")
    print(f"{'═' * 60}")

    df = pd.read_parquet(path)
    n_original = len(df)

    if n_original == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    print(f"  Nombre initial de reviews : {n_original:,}")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : Suppression des reviews sans note valide
    # ──────────────────────────────────────────────────────────────────
    # Deux cas possibles dans le jeu de données Amazon :
    #   - Le champ 'rating' est absent du JSON original → NaN en Pandas
    #   - Le champ 'rating' vaut 0, ce qui est hors de l'échelle 1–5
    # Dans les deux cas, la review est inutilisable pour un système de
    # recommandation, on la supprime.
    # ──────────────────────────────────────────────────────────────────

    n_nan_rating = df["rating"].isna().sum()
    n_zero_rating = (df["rating"] == 0).sum()

    df = df.dropna(subset=["rating"])
    df = df[df["rating"] > 0]
    n_after_rating = len(df)
    n_dropped_rating = n_original - n_after_rating

    print(f"\n  ── Étape 1 : Nettoyage des notes (rating) ──")
    print(f"     Reviews avec rating NaN   : {n_nan_rating:,}")
    print(f"     Reviews avec rating == 0  : {n_zero_rating:,}")
    print(f"     Total supprimées          : {n_dropped_rating:,}")
    print(f"     Reviews restantes         : {n_after_rating:,}")
    if n_dropped_rating == 0:
        print(f"     ✓ Aucune review sans note — le jeu de données est propre sur ce critère.")
    else:
        print(f"     → {n_dropped_rating / n_original * 100:.2f}% des reviews n'avaient pas de note valide.")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : Filtrage des timestamps invalides
    # ──────────────────────────────────────────────────────────────────
    # Les timestamps sont stockés en millisecondes Unix (ms depuis
    # le 1er janvier 1970). On les convertit en datetime pour vérifier
    # qu'ils tombent dans une plage plausible :
    #   - Borne inférieure : 1er juillet 1995 (fondation d'Amazon)
    #   - Borne supérieure : 31 décembre 2025
    #
    # errors="coerce" transforme les valeurs non convertibles en NaT
    # (Not a Time), qui sont ensuite exclues par le filtre between().
    #
    # Cas détectés :
    #   - Timestamps nuls ou manquants → NaT après conversion
    #   - Timestamps négatifs → dates avant 1970, hors plage
    #   - Timestamps en secondes au lieu de millisecondes → dates
    #     autour de 1970, également hors plage
    #   - Timestamps dans le futur → données erronées
    # ──────────────────────────────────────────────────────────────────

    n_null_ts = df["timestamp"].isna().sum()
    df["_date"] = pd.to_datetime(df["timestamp"], unit="ms", errors="coerce")
    n_nat = df["_date"].isna().sum()
    n_before_min = (df["_date"] < MIN_DATE).sum()
    n_after_max = (df["_date"] > MAX_DATE).sum()

    df = df[df["_date"].between(MIN_DATE, MAX_DATE)]
    df.drop(columns=["_date"], inplace=True)
    n_after_ts = len(df)
    n_dropped_ts = n_after_rating - n_after_ts

    print(f"\n  ── Étape 2 : Nettoyage des timestamps ──")
    print(f"     Timestamps nuls/manquants          : {n_null_ts:,}")
    print(f"     Non convertibles (→ NaT)           : {n_nat:,}")
    print(f"     Avant {MIN_DATE.date()} (pré-Amazon) : {n_before_min:,}")
    print(f"     Après {MAX_DATE.date()} (futur)      : {n_after_max:,}")
    print(f"     Total supprimées                   : {n_dropped_ts:,}")
    print(f"     Reviews restantes                  : {n_after_ts:,}")
    if n_dropped_ts == 0:
        print(f"     ✓ Tous les timestamps sont valides — aucune suppression nécessaire.")
    else:
        print(f"     → {n_dropped_ts / n_after_rating * 100:.2f}% des reviews avaient un timestamp invalide.")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : Conversion du rating en float
    # ──────────────────────────────────────────────────────────────────
    # Selon la méthode d'écriture du Parquet, le rating peut être stocké
    # en int64, int32, ou déjà en float. On force float64 pour :
    #   - Garantir la cohérence entre tous les échantillons
    #   - Éviter les erreurs de division entière dans les calculs de
    #     moyennes et corrélations en aval
    # ──────────────────────────────────────────────────────────────────

    dtype_before = df["rating"].dtype
    df["rating"] = df["rating"].astype(float)
    dtype_after = df["rating"].dtype

    print(f"\n  ── Étape 3 : Conversion du type de rating ──")
    print(f"     Type avant conversion : {dtype_before}")
    print(f"     Type après conversion : {dtype_after}")
    if dtype_before == dtype_after:
        print(f"     ✓ Le rating était déjà en {dtype_after} — aucune conversion nécessaire.")
    else:
        print(f"     → Converti de {dtype_before} vers {dtype_after}.")

    # ══════════════════════════════════════════════════════════════════
    # RÉSUMÉ FINAL
    # ══════════════════════════════════════════════════════════════════

    n_final = len(df)
    n_total_dropped = n_original - n_final

    print(f"\n  {'─' * 56}")
    print(f"  RÉSUMÉ — {path}")
    print(f"  {'─' * 56}")
    print(f"     Reviews initiales     : {n_original:,}")
    print(f"     Supprimées (rating)   : {n_dropped_rating:,}")
    print(f"     Supprimées (timestamp): {n_dropped_ts:,}")
    print(f"     Reviews finales       : {n_final:,}")
    print(f"     Taux de rétention     : {n_final / n_original * 100:.2f}%")
    print(f"     Utilisateurs restants : {df['user_id'].nunique():,}")
    print(f"     Livres restants       : {df['parent_asin'].nunique():,}")
    print(f"  {'─' * 56}")

    # ── Sauvegarde (décommenter quand prêt) ────────────────────────
    df.to_parquet(path, index=False)
    print(f"  ✓ Fichier nettoyé sauvegardé : {path}")


════════════════════════════════════════════════════════════
  Fichier : sample-cudf-claude/sample_gpu_active_users.parquet
════════════════════════════════════════════════════════════
  Nombre initial de reviews : 499,532

  ── Étape 1 : Nettoyage des notes (rating) ──
     Reviews avec rating NaN   : 0
     Reviews avec rating == 0  : 0
     Total supprimées          : 0
     Reviews restantes         : 499,532
     ✓ Aucune review sans note — le jeu de données est propre sur ce critère.

  ── Étape 2 : Nettoyage des timestamps ──
     Timestamps nuls/manquants          : 0
     Non convertibles (→ NaT)           : 0
     Avant 1995-07-01 (pré-Amazon) : 0
     Après 2025-12-31 (futur)      : 0
     Total supprimées                   : 0
     Reviews restantes                  : 499,532
     ✓ Tous les timestamps sont valides — aucune suppression nécessaire.

  ── Étape 3 : Conversion du type de rating ──
     Type avant conversion : float64
     Type après conversion : float64
     

KeyboardInterrupt: 

## Filtering ##

- Minimum number of ratings per user: 10–20
- Minimum number of ratings per book: 5–10
- Recalculate the sparsity rate after filtering

In [ ]:
import glob
import os
import pandas as pd

# ══════════════════════════════════════════════════════════════════════════
# FILTRAGE PAR SEUILS D'ACTIVITÉ (Threshold Filtering)
#
# Objectif : éliminer les utilisateurs et les livres ayant trop peu
# d'interactions. En filtrage collaboratif, un utilisateur avec 1–2 notes
# ne fournit aucun signal exploitable (problème du « cold start »), et un
# livre noté par un seul lecteur ne peut pas être recommandé par similarité.
#
# ── Choix des seuils ──────────────────────────────────────────────────────
#
# MIN_RATINGS_USER = 20
#   Notre échantillon « cudf-claude » a été construit en ne gardant que
#   les utilisateurs ayant ≥ 20 reviews dans le jeu COMPLET (~27M).
#   Toutefois, après le nettoyage précédent (rating/timestamps) et après
#   le filtrage des livres ci-dessous, certains utilisateurs pourraient
#   passer sous ce seuil. On réapplique donc 20 pour maintenir la
#   cohérence avec le critère d'échantillonnage initial.
#   ► Justification : 20 est le seuil standard dans la littérature
#     sur les systèmes de recommandation (cf. Koren 2008, He et al. 2017).
#     Il garantit un profil utilisateur assez riche pour que les algorithmes
#     de voisinage (kNN) ou de factorisation (SVD/ALS) convergent.
#
# MIN_RATINGS_BOOK = 5
#   Un livre noté par < 5 lecteurs est statistiquement « invisible » :
#   sa moyenne est instable et la co-occurrence avec d'autres livres est
#   trop faible pour contribuer aux vecteurs de similarité.
#   ► Justification : le seuil de 5 est couramment utilisé dans les
#     benchmarks Amazon (McAuley et al. 2015, Ni et al. 2019).
#     On privilégie 5 plutôt que 10 pour conserver un catalogue plus
#     large et limiter la perte de diversité (long tail).
#
# ── Filtrage itératif ─────────────────────────────────────────────────────
#
# IMPORTANT : le filtrage n'est PAS une opération unique.
# Supprimer des livres rares peut faire descendre certains utilisateurs
# sous le seuil, et inversement. On boucle donc jusqu'à convergence
# (plus aucune suppression à effectuer). En pratique, 2–4 itérations
# suffisent.
# ══════════════════════════════════════════════════════════════════════════

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))

MIN_RATINGS_USER = 20
MIN_RATINGS_BOOK = 5

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:
        print(f"  ⚠ Fichier ignoré : {path} (trop petit)")
        continue

    print(f"\n{'═' * 70}")
    print(f"  Fichier : {path}")
    print(f"{'═' * 70}")

    df = pd.read_parquet(path)
    n_before_filter = len(df)
    u_before = df["user_id"].nunique()
    b_before = df["parent_asin"].nunique()

    if n_before_filter == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    # ──────────────────────────────────────────────────────────────────
    # Calcul de la sparsité AVANT filtrage
    # ──────────────────────────────────────────────────────────────────
    # La sparsité mesure le pourcentage de cases vides dans la matrice
    # utilisateur × livre. Formule :
    #   sparsité = 1 − |R| / (|U| × |I|)
    # où |R| = nombre de ratings, |U| = nb d'utilisateurs, |I| = nb de livres.
    #
    # Une sparsité de 99.99% signifie que seule 0.01% de la matrice est
    # remplie — c'est typique des grands jeux de données de recommandation.
    # ──────────────────────────────────────────────────────────────────

    sparsity_before = 1 - n_before_filter / (u_before * b_before)

    print(f"\n  ── État AVANT filtrage ──")
    print(f"     Reviews       : {n_before_filter:,}")
    print(f"     Utilisateurs  : {u_before:,}")
    print(f"     Livres        : {b_before:,}")
    print(f"     Sparsité      : {sparsity_before * 100:.4f}%")
    print(f"     Densité       : {(1 - sparsity_before) * 100:.6f}%")

    # ──────────────────────────────────────────────────────────────────
    # Diagnostic pré-filtrage : distribution de l'activité
    # ──────────────────────────────────────────────────────────────────
    # On affiche combien d'utilisateurs/livres seraient éliminés pour
    # donner une idée de l'impact avant de filtrer.
    # ──────────────────────────────────────────────────────────────────

    ratings_per_user = df.groupby("user_id").size()
    ratings_per_book = df.groupby("parent_asin").size()

    n_users_below = (ratings_per_user < MIN_RATINGS_USER).sum()
    n_books_below = (ratings_per_book < MIN_RATINGS_BOOK).sum()

    print(f"\n  ── Diagnostic pré-filtrage ──")
    print(f"     Seuil utilisateur : ≥ {MIN_RATINGS_USER} ratings")
    print(f"     Seuil livre       : ≥ {MIN_RATINGS_BOOK} ratings")
    print(f"     Utilisateurs sous le seuil : {n_users_below:,} / {u_before:,}"
          f" ({n_users_below / u_before * 100:.1f}%)")
    print(f"     Livres sous le seuil       : {n_books_below:,} / {b_before:,}"
          f" ({n_books_below / b_before * 100:.1f}%)")

    # ──────────────────────────────────────────────────────────────────
    # Filtrage itératif jusqu'à convergence
    # ──────────────────────────────────────────────────────────────────
    # À chaque itération :
    #   1. On supprime les livres ayant < MIN_RATINGS_BOOK notes
    #   2. On supprime les utilisateurs ayant < MIN_RATINGS_USER notes
    #   3. Si rien n'a changé → on a convergé, on arrête
    #
    # On limite à 20 itérations par sécurité (jamais atteint en pratique).
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  ── Filtrage itératif ──")
    MAX_ITER = 20

    for iteration in range(1, MAX_ITER + 1):
        n_start = len(df)

        # Filtrer les livres avec trop peu de notes
        book_counts = df.groupby("parent_asin").size()
        books_ok = book_counts[book_counts >= MIN_RATINGS_BOOK].index
        df = df[df["parent_asin"].isin(books_ok)]
        n_after_books = len(df)
        dropped_books_reviews = n_start - n_after_books

        # Filtrer les utilisateurs avec trop peu de notes
        user_counts = df.groupby("user_id").size()
        users_ok = user_counts[user_counts >= MIN_RATINGS_USER].index
        df = df[df["user_id"].isin(users_ok)]
        n_after_users = len(df)
        dropped_users_reviews = n_after_books - n_after_users

        total_dropped = n_start - n_after_users

        print(f"     Itération {iteration:>2d} : "
              f"−{dropped_books_reviews:,} (livres) "
              f"−{dropped_users_reviews:,} (users) "
              f"→ {n_after_users:,} reviews restantes")

        if total_dropped == 0:
            print(f"     ✓ Convergence atteinte à l'itération {iteration}.")
            break
    else:
        print(f"     ⚠ Limite de {MAX_ITER} itérations atteinte sans convergence.")

    # ──────────────────────────────────────────────────────────────────
    # Calcul de la sparsité APRÈS filtrage
    # ──────────────────────────────────────────────────────────────────

    n_after_filter = len(df)
    u_after = df["user_id"].nunique()
    b_after = df["parent_asin"].nunique()

    if u_after > 0 and b_after > 0:
        sparsity_after = 1 - n_after_filter / (u_after * b_after)
    else:
        sparsity_after = 1.0

    # ──────────────────────────────────────────────────────────────────
    # Résumé comparatif AVANT / APRÈS
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ DU FILTRAGE — {path}")
    print(f"  {'─' * 66}")
    print(f"  {'':30s} {'AVANT':>14s}   {'APRÈS':>14s}   {'Δ':>10s}")
    print(f"  {'─' * 66}")

    delta_r = n_after_filter - n_before_filter
    delta_u = u_after - u_before
    delta_b = b_after - b_before

    print(f"  {'Reviews':30s} {n_before_filter:>14,}   {n_after_filter:>14,}   {delta_r:>+10,}")
    print(f"  {'Utilisateurs':30s} {u_before:>14,}   {u_after:>14,}   {delta_u:>+10,}")
    print(f"  {'Livres':30s} {b_before:>14,}   {b_after:>14,}   {delta_b:>+10,}")
    print(f"  {'Sparsité (%)':30s} {sparsity_before * 100:>13.4f}%   {sparsity_after * 100:>13.4f}%")
    print(f"  {'Densité (%)':30s} {(1 - sparsity_before) * 100:>13.6f}%   {(1 - sparsity_after) * 100:>13.6f}%")
    print(f"  {'─' * 66}")

    # ── Interprétation automatique des résultats ──────────────────────

    pct_reviews_kept = n_after_filter / n_before_filter * 100 if n_before_filter > 0 else 0
    pct_users_kept = u_after / u_before * 100 if u_before > 0 else 0
    pct_books_kept = b_after / b_before * 100 if b_before > 0 else 0
    density_gain = ((1 - sparsity_after) / (1 - sparsity_before) - 1) * 100 if sparsity_before < 1 else 0

    print(f"\n  ── Interprétation ──")
    print(f"     Taux de rétention des reviews       : {pct_reviews_kept:.1f}%")
    print(f"     Taux de rétention des utilisateurs   : {pct_users_kept:.1f}%")
    print(f"     Taux de rétention des livres         : {pct_books_kept:.1f}%")
    print(f"     Gain de densité                      : ×{density_gain / 100 + 1:.1f} ({density_gain:+.1f}%)")
    print()
    print(f"     La matrice U×I est passée de {u_before:,}×{b_before:,} = {u_before * b_before:,} cases")
    print(f"     à {u_after:,}×{b_after:,} = {u_after * b_after:,} cases.")
    print(f"     En éliminant les livres rares (< {MIN_RATINGS_BOOK} notes) et les utilisateurs")
    print(f"     peu actifs (< {MIN_RATINGS_USER} notes), on concentre le signal utile")
    print(f"     sur un sous-ensemble plus dense, ce qui améliore directement")
    print(f"     la qualité des recommandations par filtrage collaboratif.")

    # ── Sauvegarde (décommenter quand prêt) ────────────────────────
    df.to_parquet(path, index=False)
    print(f"\n  ✓ Fichier filtré sauvegardé : {path}")


══════════════════════════════════════════════════════════════════════
  Fichier : sample-cudf-claude/sample_gpu_active_users.parquet
══════════════════════════════════════════════════════════════════════

  ── État AVANT filtrage ──
     Reviews       : 2,442,267
     Utilisateurs  : 50,000
     Livres        : 1,042,030
     Sparsité      : 99.9953%
     Densité       : 0.004688%

  ── Diagnostic pré-filtrage ──
     Seuil utilisateur : ≥ 20 ratings
     Seuil livre       : ≥ 5 ratings
     Utilisateurs sous le seuil : 0 / 50,000 (0.0%)
     Livres sous le seuil       : 945,677 / 1,042,030 (90.8%)

  ── Filtrage itératif ──
     Itération  1 : −1,363,241 (livres) −332,187 (users) → 746,839 reviews restantes
     Itération  2 : −109,708 (livres) −60,021 (users) → 577,110 reviews restantes
     Itération  3 : −28,260 (livres) −17,862 (users) → 530,988 reviews restantes
     Itération  4 : −9,995 (livres) −6,782 (users) → 514,211 reviews restantes
     Itération  5 : −3,854 (livres) −2

## User-Item Matrix ##

- Build the matrix R ∈ R^(|U|×|I|) where r_(u,i) represents the rating of user u for book i

In [ ]:
import glob
import os
import time
import sys
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, coo_matrix

# ══════════════════════════════════════════════════════════════════════════
# CONSTRUCTION DE LA MATRICE UTILISATEUR-LIVRE (User-Item Matrix)
#
# On construit la matrice R ∈ ℝ^(|U| × |I|) où r(u,i) = note de
# l'utilisateur u pour le livre i, et 0 si l'utilisateur n'a pas noté
# ce livre.
#
# ── Pourquoi une matrice creuse (sparse) ? ────────────────────────────────
#
# Même après filtrage, la matrice est extrêmement creuse :
#   - Si |U| = 10 000 et |I| = 20 000, la matrice dense ferait
#     10 000 × 20 000 = 200 000 000 entrées × 8 octets = ~1.5 Go
#   - Or seules ~500 000 cases sont remplies (< 0.25% de la matrice)
#   - En format CSR, on ne stocke QUE les valeurs non nulles + leurs
#     indices → ~500 000 × 12 octets ≈ 6 Mo (250× moins de mémoire)
#
# ── Format CSR (Compressed Sparse Row) ────────────────────────────────────
#
# scipy.sparse.csr_matrix stocke la matrice via 3 tableaux :
#   - data[]    : les valeurs non nulles (les ratings)
#   - indices[] : l'indice de colonne de chaque valeur
#   - indptr[]  : pour chaque ligne i, data[indptr[i]:indptr[i+1]]
#                 contient les valeurs de la ligne i
#
# Avantages du CSR :
#   - Accès rapide par ligne (O(1) pour récupérer tous les ratings d'un user)
#   - Multiplication matrice-vecteur efficace (cœur de SVD/ALS)
#   - Compatible avec scikit-learn, surprise, implicit, etc.
#
# ── Stratégie d'encodage des identifiants ─────────────────────────────────
#
# Les user_id et parent_asin sont des chaînes de caractères (ex:
# "AHJKFDS83KD", "B00005N5PF"). Il faut les convertir en indices
# entiers 0...|U|-1 et 0...|I|-1 pour indexer la matrice.
#
# Approche classique (dict Python) :
#   user_map = {uid: i for i, uid in enumerate(df["user_id"].unique())}
#   → Lent pour > 100k identifiants (boucle Python pure)
#
# Approche optimisée (pd.factorize) :
#   codes, uniques = pd.factorize(df["user_id"])
#   → Implémenté en C dans Pandas, ~10-50× plus rapide
#   → Produit directement les indices entiers et la table de correspondance
#
# Les deux approches produisent une matrice CSR identique.
# ══════════════════════════════════════════════════════════════════════════

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))

# On stocke les matrices et mappings dans un dictionnaire pour usage ultérieur
matrices = {}

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:
        continue

    print(f"\n{'═' * 70}")
    print(f"  Matrice utilisateur-livre : {path}")
    print(f"{'═' * 70}")

    df = pd.read_parquet(path)

    if len(df) == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : Encodage des identifiants avec pd.factorize
    # ──────────────────────────────────────────────────────────────────
    # pd.factorize attribue un entier consécutif (0, 1, 2, ...) à chaque
    # valeur unique, dans l'ordre de première apparition.
    # Retourne :
    #   - codes : array d'entiers (même longueur que la colonne)
    #   - uniques : array des valeurs originales (table de correspondance)
    #
    # Ainsi : uniques[codes[i]] == df["user_id"].iloc[i]  (bijectif)
    # ──────────────────────────────────────────────────────────────────

    t0 = time.perf_counter()

    user_codes, user_ids = pd.factorize(df["user_id"], sort=False)
    item_codes, item_ids = pd.factorize(df["parent_asin"], sort=False)
    ratings = df["rating"].values.astype(np.float32)

    n_users = len(user_ids)
    n_items = len(item_ids)
    n_ratings = len(ratings)

    t_encode = time.perf_counter() - t0

    print(f"\n  ── Étape 1 : Encodage des identifiants (pd.factorize) ──")
    print(f"     Utilisateurs encodés : {n_users:,}  (indices 0 à {n_users - 1:,})")
    print(f"     Livres encodés       : {n_items:,}  (indices 0 à {n_items - 1:,})")
    print(f"     Ratings à insérer    : {n_ratings:,}")
    print(f"     Temps d'encodage     : {t_encode * 1000:.1f} ms")
    print(f"     → pd.factorize est implémenté en C dans Pandas ; un dict")
    print(f"       Python serait ~10-50× plus lent sur {n_ratings:,} entrées.")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : Construction de la matrice CSR
    # ──────────────────────────────────────────────────────────────────
    # On passe par le constructeur COO (Coordinate format) de scipy :
    #   csr_matrix((data, (row, col)), shape=(n_rows, n_cols))
    #
    # En interne, scipy convertit automatiquement en CSR.
    # Si un même couple (u, i) apparaît plusieurs fois (doublons),
    # scipy ADDITIONNE les valeurs. On vérifie et gère ce cas.
    #
    # Note : on utilise float32 au lieu de float64 pour les ratings.
    # Les notes étant des entiers 1–5, float32 (7 décimales) est
    # largement suffisant et divise l'empreinte mémoire par 2.
    # ──────────────────────────────────────────────────────────────────

    t1 = time.perf_counter()

    # Vérification des doublons (un utilisateur ayant noté 2 fois le même livre)
    n_duplicates = df.duplicated(subset=["user_id", "parent_asin"]).sum()

    if n_duplicates > 0:
        # En cas de doublons, on garde la note la plus récente (dernière)
        print(f"\n  ⚠ {n_duplicates:,} doublons détectés (même utilisateur + même livre)")
        print(f"    → On conserve la note la plus récente (dernière occurrence).")
        df = df.drop_duplicates(subset=["user_id", "parent_asin"], keep="last")
        user_codes, user_ids = pd.factorize(df["user_id"], sort=False)
        item_codes, item_ids = pd.factorize(df["parent_asin"], sort=False)
        ratings = df["rating"].values.astype(np.float32)
        n_users = len(user_ids)
        n_items = len(item_ids)
        n_ratings = len(ratings)

    R = csr_matrix(
        (ratings, (user_codes, item_codes)),
        shape=(n_users, n_items),
        dtype=np.float32,
    )

    t_build = time.perf_counter() - t1

    print(f"\n  ── Étape 2 : Construction de la matrice CSR ──")
    print(f"     Dimensions           : {R.shape[0]:,} × {R.shape[1]:,}")
    print(f"     Entrées non nulles   : {R.nnz:,}")
    print(f"     Doublons (u, i)      : {n_duplicates:,}")
    print(f"     Temps de construction: {t_build * 1000:.1f} ms")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : Analyse mémoire et validation
    # ──────────────────────────────────────────────────────────────────
    # On compare l'empreinte mémoire de la matrice creuse à celle
    # qu'aurait une matrice dense de mêmes dimensions.
    # ──────────────────────────────────────────────────────────────────

    mem_data = R.data.nbytes
    mem_indices = R.indices.nbytes
    mem_indptr = R.indptr.nbytes
    mem_sparse_total = mem_data + mem_indices + mem_indptr

    mem_dense = n_users * n_items * np.dtype(np.float32).itemsize

    sparsity = 1 - R.nnz / (n_users * n_items)

    print(f"\n  ── Étape 3 : Analyse mémoire ──")
    print(f"     Matrice creuse (CSR) :")
    print(f"       data[]    ({R.data.dtype})  : {mem_data / 1024**2:>8.2f} Mo  ({R.nnz:,} valeurs)")
    print(f"       indices[] ({R.indices.dtype}) : {mem_indices / 1024**2:>8.2f} Mo  (indices de colonnes)")
    print(f"       indptr[]  ({R.indptr.dtype}) : {mem_indptr / 1024**2:>8.2f} Mo  ({n_users + 1:,} pointeurs de lignes)")
    print(f"       TOTAL CSR            : {mem_sparse_total / 1024**2:>8.2f} Mo")
    print(f"     Matrice dense équivalente :")
    print(f"       {n_users:,} × {n_items:,} × 4 octets : {mem_dense / 1024**2:>8.2f} Mo")
    print(f"     Ratio de compression       : {mem_dense / mem_sparse_total:>8.1f}×")
    print(f"     → La représentation CSR est {mem_dense / mem_sparse_total:.0f} fois plus compacte")
    print(f"       que la matrice dense pour une sparsité de {sparsity * 100:.2f}%.")

    # ──────────────────────────────────────────────────────────────────
    # Validation de la matrice
    # ──────────────────────────────────────────────────────────────────
    # On vérifie que les ratings dans la matrice correspondent bien
    # aux ratings originaux, et que les bornes sont respectées.
    # ──────────────────────────────────────────────────────────────────

    min_val = R.data.min()
    max_val = R.data.max()
    mean_val = R.data.mean()

    print(f"\n  ── Validation ──")
    print(f"     Rating min dans R  : {min_val:.1f}  (attendu : 1.0)")
    print(f"     Rating max dans R  : {max_val:.1f}  (attendu : 5.0)")
    print(f"     Rating moyen       : {mean_val:.2f}")
    print(f"     Nb ratings / user  : min={np.diff(R.indptr).min()}, "
          f"max={np.diff(R.indptr).max()}, "
          f"moy={np.diff(R.indptr).mean():.1f}")

    if 1.0 <= min_val and max_val <= 5.0:
        print(f"     ✓ Tous les ratings sont dans l'intervalle [1, 5] — matrice valide.")
    else:
        print(f"     ⚠ ATTENTION : des ratings hors de [1, 5] détectés !")

    # ──────────────────────────────────────────────────────────────────
    # Stockage pour usage ultérieur
    # ──────────────────────────────────────────────────────────────────
    # On conserve la matrice R, ainsi que les tables de correspondance
    # (user_ids, item_ids) qui permettent de retrouver les identifiants
    # originaux à partir des indices de la matrice.
    #
    # Exemple d'utilisation :
    #   user_ids[42]  → "AHJKFDS83KD"   (ID Amazon de l'utilisateur 42)
    #   item_ids[7]   → "B00005N5PF"     (ASIN du livre en colonne 7)
    #   R[42, 7]      → 4.0             (note donnée par cet utilisateur)
    # ──────────────────────────────────────────────────────────────────

    matrices[path] = {
        "R": R,
        "user_ids": user_ids,
        "item_ids": item_ids,
    }

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ — {path}")
    print(f"  {'─' * 66}")
    print(f"     Matrice R       : {R.shape[0]:,} utilisateurs × {R.shape[1]:,} livres")
    print(f"     Ratings stockés : {R.nnz:,}")
    print(f"     Sparsité        : {sparsity * 100:.2f}%")
    print(f"     Mémoire CSR     : {mem_sparse_total / 1024**2:.2f} Mo")
    print(f"     Mémoire dense   : {mem_dense / 1024**2:.2f} Mo (économie : {mem_dense / mem_sparse_total:.0f}×)")
    print(f"     Temps total     : {(t_encode + t_build) * 1000:.1f} ms")
    print(f"  {'─' * 66}")

print(f"\n✓ {len(matrices)} matrice(s) construite(s) et stockée(s) dans `matrices`.")


══════════════════════════════════════════════════════════════════════
  Matrice utilisateur-livre : sample-cudf-claude/sample_gpu_active_users.parquet
══════════════════════════════════════════════════════════════════════

  ── Étape 1 : Encodage des identifiants (pd.factorize) ──
     Utilisateurs encodés : 10,714  (indices 0 à 10,713)
     Livres encodés       : 43,926  (indices 0 à 43,925)
     Ratings à insérer    : 499,532
     Temps d'encodage     : 28.2 ms
     → pd.factorize est implémenté en C dans Pandas ; un dict
       Python serait ~10-50× plus lent sur 499,532 entrées.

  ⚠ 9,115 doublons détectés (même utilisateur + même livre)
    → On conserve la note la plus récente (dernière occurrence).

  ── Étape 2 : Construction de la matrice CSR ──
     Dimensions           : 10,714 × 43,926
     Entrées non nulles   : 490,417
     Doublons (u, i)      : 9,115
     Temps de construction: 124.5 ms

  ── Étape 3 : Analyse mémoire ──
     Matrice creuse (CSR) :
       data[]    (

## Train/Test Split ## 

Split the data into:

    - Training set: 80% of ratings
    - Test set: 20% of ratings
    - Stratify by user: each user must have at least one rating in each set

In [ ]:
import glob
import os
import time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# ══════════════════════════════════════════════════════════════════════════
# SÉPARATION TRAIN / TEST STRATIFIÉE PAR UTILISATEUR
#
# Objectif : diviser les ratings en 80% entraînement / 20% test,
# en garantissant que CHAQUE utilisateur possède au moins 1 rating
# dans chaque ensemble.
#
# ── Pourquoi stratifier par utilisateur ? ─────────────────────────────────
#
# En recommandation, on évalue la capacité du modèle à prédire les
# goûts d'un utilisateur CONNU à partir de son historique partiel.
# Si un utilisateur n'a aucun rating dans le train, le modèle ne peut
# rien apprendre de lui (cold start total). Si un utilisateur n'a
# aucun rating dans le test, on ne peut pas mesurer la qualité des
# prédictions le concernant.
#
# Un split aléatoire global (sans stratification) risquerait de
# concentrer tous les ratings d'un utilisateur rare dans un seul
# ensemble — d'où la nécessité de stratifier.
#
# ── Contrainte pour les utilisateurs avec peu de ratings ──────────────────
#
# Pour un utilisateur avec n ratings et un ratio test de 20% :
#   n_test  = max(1, floor(n × 0.2))   — au moins 1 dans le test
#   n_test  = min(n_test, n − 1)        — au moins 1 dans le train
#
# Cas limites :
#   n = 2 → n_test = 1, n_train = 1  (split 50/50, inévitable)
#   n = 5 → n_test = 1, n_train = 4  (split 80/20)
#   n = 10 → n_test = 2, n_train = 8  (split exact 80/20)
#   n = 50 → n_test = 10, n_train = 40  (split exact 80/20)
#
# Les utilisateurs avec n = 2 auront un split plus généreux pour le
# test (50/50 au lieu de 80/20), ce qui est le prix à payer pour
# respecter la contrainte « au moins 1 dans chaque ensemble ».
#
# ── Approche vectorisée (sans boucle Python sur les utilisateurs) ─────────
#
# L'approche naïve serait de boucler sur chaque utilisateur et de
# faire un random.sample par utilisateur → O(|U|) itérations Python,
# lent pour |U| > 10 000.
#
# Notre approche vectorisée :
#   1. Attribuer un nombre aléatoire à chaque rating
#   2. Trier par (user_id, aléatoire) pour mélanger intra-utilisateur
#   3. Calculer la position cumulative (rank) de chaque rating au sein
#      de son utilisateur via groupby().cumcount()
#   4. Calculer n_test pour chaque utilisateur de manière vectorisée
#   5. Les derniers n_test ratings de chaque utilisateur vont dans le
#      test, le reste dans le train
#
# Complexité : O(|R| log |R|) pour le tri, sans aucune boucle Python.
# Sur ~500 000 ratings, cela prend < 500 ms.
# ══════════════════════════════════════════════════════════════════════════

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))

TRAIN_RATIO = 0.80
TEST_RATIO = 1 - TRAIN_RATIO
SEED = 42

splits = {}

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:
        continue

    print(f"\n{'═' * 70}")
    print(f"  Split Train/Test : {path}")
    print(f"{'═' * 70}")

    df = pd.read_parquet(path)

    if len(df) == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    # Dédoublonner (même logique que la cellule matrice)
    n_dup = df.duplicated(subset=["user_id", "parent_asin"]).sum()
    if n_dup > 0:
        df = df.drop_duplicates(subset=["user_id", "parent_asin"], keep="last")
        print(f"  {n_dup:,} doublons supprimés (même logique que la matrice CSR)")

    n_total = len(df)
    n_users = df["user_id"].nunique()
    n_items = df["parent_asin"].nunique()

    print(f"  Ratings : {n_total:,}  |  Utilisateurs : {n_users:,}  |  Livres : {n_items:,}")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : Mélange aléatoire intra-utilisateur (vectorisé)
    # ──────────────────────────────────────────────────────────────────
    # On attribue un nombre aléatoire à chaque ligne, puis on trie
    # par (user_id, aléatoire). Cela revient à faire un shuffle
    # indépendant des ratings de chaque utilisateur, mais en une
    # seule opération de tri sur tout le DataFrame.
    # ──────────────────────────────────────────────────────────────────

    t0 = time.perf_counter()

    rng = np.random.RandomState(SEED)
    df["_rand"] = rng.random(len(df))
    df = df.sort_values(["user_id", "_rand"]).reset_index(drop=True)

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : Calcul vectorisé de la position et du seuil de split
    # ──────────────────────────────────────────────────────────────────
    # Pour chaque utilisateur :
    #   - cumcount() : position 0, 1, 2, ... au sein de ses ratings
    #   - transform("count") : nombre total de ratings de cet utilisateur
    #   - n_test = max(1, floor(total × 0.2)), borné à total − 1
    #
    # Un rating est dans le TEST si sa position ≥ (total − n_test),
    # c'est-à-dire s'il fait partie des derniers n_test ratings
    # (après mélange aléatoire).
    # ──────────────────────────────────────────────────────────────────

    df["_pos"] = df.groupby("user_id").cumcount()
    df["_total"] = df.groupby("user_id")["_pos"].transform("count")

    n_test_per_user = np.floor(df["_total"].values * TEST_RATIO).astype(int)
    n_test_per_user = np.maximum(n_test_per_user, 1)          # au moins 1 en test
    n_test_per_user = np.minimum(n_test_per_user, df["_total"].values - 1)  # au moins 1 en train

    df["_n_test"] = n_test_per_user
    df["is_test"] = df["_pos"] >= (df["_total"] - df["_n_test"])

    train_df = df[~df["is_test"]].drop(columns=["_rand", "_pos", "_total", "_n_test", "is_test"])
    test_df = df[df["is_test"]].drop(columns=["_rand", "_pos", "_total", "_n_test", "is_test"])

    t_split = time.perf_counter() - t0

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : Validation de la contrainte de stratification
    # ──────────────────────────────────────────────────────────────────
    # On vérifie que CHAQUE utilisateur a au moins 1 rating dans le
    # train ET dans le test. C'est la contrainte fondamentale du split.
    # ──────────────────────────────────────────────────────────────────

    users_in_train = set(train_df["user_id"].unique())
    users_in_test = set(test_df["user_id"].unique())
    users_only_train = users_in_train - users_in_test
    users_only_test = users_in_test - users_in_train
    users_in_both = users_in_train & users_in_test

    print(f"\n  ── Étape 1-2 : Split vectorisé (seed={SEED}) ──")
    print(f"     Temps de calcul : {t_split * 1000:.1f} ms")
    print(f"     Ratio demandé   : {TRAIN_RATIO:.0%} train / {TEST_RATIO:.0%} test")

    actual_train_ratio = len(train_df) / n_total
    actual_test_ratio = len(test_df) / n_total

    print(f"     Ratio effectif  : {actual_train_ratio:.2%} train / {actual_test_ratio:.2%} test")
    if abs(actual_train_ratio - TRAIN_RATIO) > 0.02:
        print(f"     → Écart de {abs(actual_train_ratio - TRAIN_RATIO):.1%} par rapport au ratio cible.")
        print(f"       Cela est dû aux utilisateurs avec peu de ratings (n=2 ou 3)")
        print(f"       pour lesquels on est forcé de donner 1 rating au test,")
        print(f"       ce qui « sur-représente » légèrement le test.")
    else:
        print(f"     → Ratio respecté à ±2% près.")

    print(f"\n  ── Étape 3 : Validation de la stratification ──")
    print(f"     Utilisateurs dans train ET test  : {len(users_in_both):,}")
    print(f"     Utilisateurs SEULEMENT dans train: {len(users_only_train):,}")
    print(f"     Utilisateurs SEULEMENT dans test : {len(users_only_test):,}")

    if len(users_only_train) == 0 and len(users_only_test) == 0:
        print(f"     ✓ Contrainte respectée : chaque utilisateur a au moins 1 rating")
        print(f"       dans chaque ensemble.")
    else:
        print(f"     ⚠ VIOLATION : {len(users_only_train) + len(users_only_test)} "
              f"utilisateur(s) absent(s) d'un ensemble !")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 4 : Construction des matrices CSR train et test
    # ──────────────────────────────────────────────────────────────────
    # On réutilise pd.factorize sur l'ensemble COMPLET (train + test)
    # pour garantir que les indices utilisateur/livre sont cohérents
    # entre les deux matrices. Sinon, l'utilisateur 42 dans R_train
    # pourrait correspondre à un utilisateur différent dans R_test.
    # ──────────────────────────────────────────────────────────────────

    t1 = time.perf_counter()

    all_users = pd.concat([train_df["user_id"], test_df["user_id"]])
    all_items = pd.concat([train_df["parent_asin"], test_df["parent_asin"]])
    user_codes_all, user_ids = pd.factorize(all_users, sort=False)
    item_codes_all, item_ids = pd.factorize(all_items, sort=False)

    n_u = len(user_ids)
    n_i = len(item_ids)
    n_train = len(train_df)
    n_test = len(test_df)

    # Les n_train premiers codes correspondent au train, le reste au test
    train_user_codes = user_codes_all[:n_train]
    train_item_codes = item_codes_all[:n_train]
    test_user_codes = user_codes_all[n_train:]
    test_item_codes = item_codes_all[n_train:]

    R_train = csr_matrix(
        (train_df["rating"].values.astype(np.float32),
         (train_user_codes, train_item_codes)),
        shape=(n_u, n_i),
        dtype=np.float32,
    )

    R_test = csr_matrix(
        (test_df["rating"].values.astype(np.float32),
         (test_user_codes, test_item_codes)),
        shape=(n_u, n_i),
        dtype=np.float32,
    )

    t_build = time.perf_counter() - t1

    print(f"\n  ── Étape 4 : Matrices CSR train/test ──")
    print(f"     Dimensions communes : {n_u:,} × {n_i:,}")
    print(f"     R_train : {R_train.nnz:,} entrées  ({R_train.nnz / (n_u * n_i) * 100:.4f}% dense)")
    print(f"     R_test  : {R_test.nnz:,} entrées  ({R_test.nnz / (n_u * n_i) * 100:.4f}% dense)")
    print(f"     R_train + R_test = {R_train.nnz + R_test.nnz:,} "
          f"(total original : {n_total:,}, "
          f"après dédup : {n_train + n_test:,})")
    print(f"     Temps de construction : {t_build * 1000:.1f} ms")

    mem_train = R_train.data.nbytes + R_train.indices.nbytes + R_train.indptr.nbytes
    mem_test = R_test.data.nbytes + R_test.indices.nbytes + R_test.indptr.nbytes

    print(f"     Mémoire R_train : {mem_train / 1024**2:.2f} Mo")
    print(f"     Mémoire R_test  : {mem_test / 1024**2:.2f} Mo")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 5 : Distribution du nombre de ratings test par utilisateur
    # ──────────────────────────────────────────────────────────────────
    # On affiche la distribution pour vérifier que le split est
    # raisonnablement équilibré et que les cas extrêmes (n=2) sont
    # bien gérés.
    # ──────────────────────────────────────────────────────────────────

    test_per_user = test_df.groupby("user_id").size()
    train_per_user = train_df.groupby("user_id").size()

    print(f"\n  ── Étape 5 : Distribution du split par utilisateur ──")
    print(f"     Ratings TRAIN par utilisateur :")
    print(f"       min = {train_per_user.min()}  |  "
          f"médiane = {train_per_user.median():.0f}  |  "
          f"moyenne = {train_per_user.mean():.1f}  |  "
          f"max = {train_per_user.max()}")
    print(f"     Ratings TEST par utilisateur :")
    print(f"       min = {test_per_user.min()}  |  "
          f"médiane = {test_per_user.median():.0f}  |  "
          f"moyenne = {test_per_user.mean():.1f}  |  "
          f"max = {test_per_user.max()}")

    n_users_1_test = (test_per_user == 1).sum()
    print(f"     Utilisateurs avec exactement 1 rating test : {n_users_1_test:,}"
          f" ({n_users_1_test / n_users * 100:.1f}%)")
    print(f"     → Ce sont les utilisateurs avec ≤ 5 ratings originaux,")
    print(f"       pour lesquels floor(n × 0.2) ≤ 1.")

    # ──────────────────────────────────────────────────────────────────
    # Résumé final
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ DU SPLIT — {path}")
    print(f"  {'─' * 66}")
    print(f"  {'':30s} {'TRAIN':>14s}   {'TEST':>14s}   {'TOTAL':>10s}")
    print(f"  {'─' * 66}")
    print(f"  {'Ratings':30s} {n_train:>14,}   {n_test:>14,}   {n_train + n_test:>10,}")
    print(f"  {'Proportion':30s} {actual_train_ratio:>13.2%}   {actual_test_ratio:>13.2%}   {'100.00%':>10s}")
    print(f"  {'Utilisateurs présents':30s} {train_df['user_id'].nunique():>14,}   "
          f"{test_df['user_id'].nunique():>14,}   {n_users:>10,}")
    print(f"  {'Livres présents':30s} {train_df['parent_asin'].nunique():>14,}   "
          f"{test_df['parent_asin'].nunique():>14,}   {n_items:>10,}")
    print(f"  {'Mémoire CSR':30s} {mem_train / 1024**2:>13.2f}Mo   "
          f"{mem_test / 1024**2:>13.2f}Mo")
    print(f"  {'─' * 66}")

    # Stockage pour cellules suivantes
    splits[path] = {
        "R_train": R_train,
        "R_test": R_test,
        "train_df": train_df,
        "test_df": test_df,
        "user_ids": user_ids,
        "item_ids": item_ids,
    }

print(f"\n✓ {len(splits)} split(s) train/test construit(s) et stocké(s) dans `splits`.")


══════════════════════════════════════════════════════════════════════
  Split Train/Test : sample-cudf-claude/sample_gpu_active_users.parquet
══════════════════════════════════════════════════════════════════════
  9,115 doublons supprimés (même logique que la matrice CSR)
  Ratings : 490,417  |  Utilisateurs : 10,714  |  Livres : 43,926

  ── Étape 1-2 : Split vectorisé (seed=42) ──
     Temps de calcul : 221.5 ms
     Ratio demandé   : 80% train / 20% test
     Ratio effectif  : 81.30% train / 18.70% test
     → Ratio respecté à ±2% près.

  ── Étape 3 : Validation de la stratification ──
     Utilisateurs dans train ET test  : 10,714
     Utilisateurs SEULEMENT dans train: 0
     Utilisateurs SEULEMENT dans test : 0
     ✓ Contrainte respectée : chaque utilisateur a au moins 1 rating
       dans chaque ensemble.

  ── Étape 4 : Matrices CSR train/test ──
     Dimensions communes : 10,714 × 43,926
     R_train : 398,731 entrées  (0.0847% dense)
     R_test  : 91,686 entrées  (0.019

### Store splitted data ###

In [10]:
import json
import os
import time
from pathlib import Path
from scipy.sparse import save_npz
import numpy as np

# ══════════════════════════════════════════════════════════════════════════
# SAUVEGARDE PERSISTANTE DES SPLITS TRAIN / TEST
#
# On écrit sur disque tous les artefacts nécessaires pour reprendre
# le travail sans avoir à ré-exécuter les cellules précédentes :
#
#   1. train.parquet / test.parquet
#      → Les DataFrames bruts (toutes les colonnes : user_id, parent_asin,
#        rating, timestamp, text, etc.). Format Parquet pour la portabilité
#        et la vitesse de lecture.
#
#   2. R_train.npz / R_test.npz
#      → Les matrices CSR au format natif scipy. Rechargement instantané
#        avec scipy.sparse.load_npz(), sans recalcul de pd.factorize
#        ni reconstruction de la matrice.
#
#   3. user_ids.npy / item_ids.npy
#      → Les tables de correspondance indice ↔ identifiant original.
#        Indispensables pour interpréter les résultats des modèles :
#          user_ids[42] → "AHJKFDS83KD" (ID Amazon)
#          item_ids[7]  → "B00005N5PF"  (ASIN du livre)
#
#   4. metadata.json
#      → Paramètres du split et statistiques clés. Permet de vérifier
#        la provenance et la cohérence des données sans les recharger.
#
# ── Structure sur disque ──────────────────────────────────────────────────
#
# Pour chaque échantillon (ex: sample-cudf-claude/), on crée un
# sous-dossier « splits/ » :
#
#   sample-cudf-claude/
#   ├── sample_gpu_active_users.parquet   ← fichier source (déjà existant)
#   └── splits/
#       ├── train.parquet                 ← 80% des ratings
#       ├── test.parquet                  ← 20% des ratings
#       ├── R_train.npz                   ← matrice CSR d'entraînement
#       ├── R_test.npz                    ← matrice CSR de test
#       ├── user_ids.npy                  ← mapping indice → user_id
#       ├── item_ids.npy                  ← mapping indice → parent_asin
#       └── metadata.json                ← paramètres et statistiques
#
# ── Pourquoi ce choix de formats ? ────────────────────────────────────────
#
# Parquet (DataFrames) :
#   - Compression columnar → 3-5× plus compact que CSV
#   - Préserve les types (float, int, string) sans ambiguïté
#   - Lisible par Pandas, Polars, DuckDB, Spark, etc.
#
# NPZ (matrices CSR) :
#   - Format natif de scipy.sparse → load_npz() reconstitue la CSR
#     en une seule instruction, sans re-factoriser les identifiants
#   - Stocke data[], indices[], indptr[] + shape en un seul fichier
#
# NPY (mappings) :
#   - Format natif NumPy, le plus rapide pour charger un array 1D
#   - allow_pickle=True nécessaire pour les arrays de strings
#
# JSON (métadonnées) :
#   - Lisible par un humain, facilement parsable
#   - Documente la provenance exacte des splits
# ══════════════════════════════════════════════════════════════════════════

for path, data in splits.items():
    sample_dir = Path(path).parent
    split_dir = sample_dir / "splits"
    split_dir.mkdir(exist_ok=True)

    print(f"\n{'═' * 70}")
    print(f"  Sauvegarde : {split_dir}/")
    print(f"{'═' * 70}")

    t0 = time.perf_counter()

    R_train = data["R_train"]
    R_test = data["R_test"]
    train_df = data["train_df"]
    test_df = data["test_df"]
    user_ids = data["user_ids"]
    item_ids = data["item_ids"]

    # ── 1. DataFrames Parquet ─────────────────────────────────────────

    train_path = split_dir / "train.parquet"
    test_path = split_dir / "test.parquet"

    train_df.to_parquet(train_path, index=False)
    test_df.to_parquet(test_path, index=False)

    size_train_pq = os.path.getsize(train_path)
    size_test_pq = os.path.getsize(test_path)

    print(f"\n  ── DataFrames Parquet ──")
    print(f"     {train_path.name:20s} : {len(train_df):>10,} lignes  |  {size_train_pq / 1024**2:>7.2f} Mo")
    print(f"     {test_path.name:20s} : {len(test_df):>10,} lignes  |  {size_test_pq / 1024**2:>7.2f} Mo")

    # ── 2. Matrices CSR (scipy NPZ) ──────────────────────────────────

    r_train_path = split_dir / "R_train.npz"
    r_test_path = split_dir / "R_test.npz"

    save_npz(r_train_path, R_train)
    save_npz(r_test_path, R_test)

    size_train_npz = os.path.getsize(r_train_path)
    size_test_npz = os.path.getsize(r_test_path)

    print(f"\n  ── Matrices CSR (NPZ) ──")
    print(f"     {r_train_path.name:20s} : {R_train.shape[0]:,}×{R_train.shape[1]:,}  "
          f"|  nnz={R_train.nnz:>10,}  |  {size_train_npz / 1024**2:>7.2f} Mo")
    print(f"     {r_test_path.name:20s} : {R_test.shape[0]:,}×{R_test.shape[1]:,}  "
          f"|  nnz={R_test.nnz:>10,}  |  {size_test_npz / 1024**2:>7.2f} Mo")

    # ── 3. Mappings (NumPy NPY) ──────────────────────────────────────

    user_path = split_dir / "user_ids.npy"
    item_path = split_dir / "item_ids.npy"

    np.save(user_path, user_ids)
    np.save(item_path, item_ids)

    size_user = os.path.getsize(user_path)
    size_item = os.path.getsize(item_path)

    print(f"\n  ── Mappings (NPY) ──")
    print(f"     {user_path.name:20s} : {len(user_ids):>10,} entrées  |  {size_user / 1024**2:>7.2f} Mo")
    print(f"     {item_path.name:20s} : {len(item_ids):>10,} entrées  |  {size_item / 1024**2:>7.2f} Mo")

    # ── 4. Métadonnées JSON ──────────────────────────────────────────

    metadata = {
        "source_file": str(path),
        "split_seed": SEED,
        "train_ratio": TRAIN_RATIO,
        "test_ratio": TEST_RATIO,
        "n_users": int(len(user_ids)),
        "n_items": int(len(item_ids)),
        "train": {
            "n_ratings": int(R_train.nnz),
            "n_users": int(train_df["user_id"].nunique()),
            "n_items": int(train_df["parent_asin"].nunique()),
            "sparsity": float(1 - R_train.nnz / (len(user_ids) * len(item_ids))),
            "file_parquet": "train.parquet",
            "file_csr": "R_train.npz",
        },
        "test": {
            "n_ratings": int(R_test.nnz),
            "n_users": int(test_df["user_id"].nunique()),
            "n_items": int(test_df["parent_asin"].nunique()),
            "sparsity": float(1 - R_test.nnz / (len(user_ids) * len(item_ids))),
            "file_parquet": "test.parquet",
            "file_csr": "R_test.npz",
        },
        "mappings": {
            "file_user_ids": "user_ids.npy",
            "file_item_ids": "item_ids.npy",
        },
    }

    meta_path = split_dir / "metadata.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print(f"\n  ── Métadonnées ──")
    print(f"     {meta_path.name:20s} : paramètres du split + statistiques")

    # ── Résumé ────────────────────────────────────────────────────────

    t_save = time.perf_counter() - t0
    total_size = size_train_pq + size_test_pq + size_train_npz + size_test_npz + size_user + size_item

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ DE LA SAUVEGARDE — {split_dir}/")
    print(f"  {'─' * 66}")
    print(f"     Fichiers écrits  : 7")
    print(f"     Taille totale    : {total_size / 1024**2:.2f} Mo")
    print(f"     Temps d'écriture : {t_save * 1000:.1f} ms")
    print(f"  {'─' * 66}")
    print(f"     Contenu du dossier :")
    for f in sorted(split_dir.iterdir()):
        print(f"       {f.name:25s}  {os.path.getsize(f) / 1024**2:>7.2f} Mo")
    print(f"  {'─' * 66}")

print(f"\n✓ Tous les splits sont sauvegardés sur disque.")
print(f"\n  Pour recharger dans un autre notebook :")
print(f"  ┌─────────────────────────────────────────────────────────────────┐")
print(f"  │  from scipy.sparse import load_npz                            │")
print(f"  │  import numpy as np, pandas as pd, json                       │")
print(f"  │                                                               │")
print(f"  │  SPLIT = 'sample-cudf-claude/splits'                          │")
print(f"  │                                                               │")
print(f"  │  train_df = pd.read_parquet(f'{{SPLIT}}/train.parquet')        │")
print(f"  │  test_df  = pd.read_parquet(f'{{SPLIT}}/test.parquet')         │")
print(f"  │  R_train  = load_npz(f'{{SPLIT}}/R_train.npz')                │")
print(f"  │  R_test   = load_npz(f'{{SPLIT}}/R_test.npz')                 │")
print(f"  │  user_ids = np.load(f'{{SPLIT}}/user_ids.npy',                │")
print(f"  │                     allow_pickle=True)                        │")
print(f"  │  item_ids = np.load(f'{{SPLIT}}/item_ids.npy',                │")
print(f"  │                     allow_pickle=True)                        │")
print(f"  │  meta     = json.load(open(f'{{SPLIT}}/metadata.json'))        │")
print(f"  └─────────────────────────────────────────────────────────────────┘")


══════════════════════════════════════════════════════════════════════
  Sauvegarde : sample-cudf-claude/splits/
══════════════════════════════════════════════════════════════════════

  ── DataFrames Parquet ──
     train.parquet        :    398,731 lignes  |   245.94 Mo
     test.parquet         :     91,686 lignes  |    59.21 Mo

  ── Matrices CSR (NPZ) ──
     R_train.npz          : 10,714×43,926  |  nnz=   398,731  |     1.08 Mo
     R_test.npz           : 10,714×43,926  |  nnz=    91,686  |     0.27 Mo

  ── Mappings (NPY) ──
     user_ids.npy         :     10,714 entrées  |     0.32 Mo
     item_ids.npy         :     43,926 entrées  |     0.55 Mo

  ── Métadonnées ──
     metadata.json        : paramètres du split + statistiques

  ──────────────────────────────────────────────────────────────────
  RÉSUMÉ DE LA SAUVEGARDE — sample-cudf-claude/splits/
  ──────────────────────────────────────────────────────────────────
     Fichiers écrits  : 7
     Taille totale    : 307.36 Mo

# Tâche 1 : Mesures de similiratés #

### Implementing the measures ###

 - Cosine Similarity
 - Pearson Correlation
 - Jaccard Similarity

##### Too long version #####

In [11]:
import glob
import os
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix, load_npz, lil_matrix, save_npz
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# ══════════════════════════════════════════════════════════════════════════
# TÂCHE 1 : MESURES DE SIMILARITÉ ENTRE UTILISATEURS
#
# On implémente trois mesures classiques de similarité pour le filtrage
# collaboratif basé sur les utilisateurs (user-based CF) :
#
#   1. Similarité cosinus (Cosine Similarity)
#   2. Corrélation de Pearson (Pearson Correlation)
#   3. Similarité de Jaccard (Jaccard Similarity)
#
# ── Contexte mathématique ─────────────────────────────────────────────────
#
# Soit R ∈ ℝ^(|U| × |I|) la matrice utilisateur-livre, où r(u,i) est
# la note de l'utilisateur u pour le livre i (0 si non noté).
#
# Pour deux utilisateurs u et v, on note :
#   I_u     = ensemble des livres notés par u
#   I_v     = ensemble des livres notés par v
#   I_{u,v} = I_u ∩ I_v = livres notés par les deux
#   r_u     = vecteur des notes de u (ligne u de R)
#   r̄_u     = moyenne des notes de u (sur I_u uniquement)
#
# ── Pourquoi ne PAS calculer la matrice complète ? ────────────────────────
#
# Pour |U| = 10 000, la matrice de similarité complète ferait
# 10 000 × 10 000 = 100 000 000 entrées × 4 octets = 400 Mo.
# C'est faisable en mémoire, MAIS :
#   - La plupart des paires (u, v) partagent très peu de livres
#   - Les similarités proches de 0 ne servent à rien pour la prédiction
#   - Stocker et manipuler une matrice dense est inutilement coûteux
#
# → On ne garde que les similarités au-dessus d'un seuil (SPARSE storage)
# → On calcule par blocs (batches) pour limiter la mémoire pic
#
# ── Pourquoi utiliser R_train et non R ? ──────────────────────────────────
#
# Les similarités sont calculées sur les données d'ENTRAÎNEMENT uniquement.
# Si on utilisait R (train + test), on « tricherait » en intégrant des
# informations de test dans le modèle (data leakage). Cela biaiserait
# l'évaluation en donnant des résultats artificiellement meilleurs.
# ══════════════════════════════════════════════════════════════════════════

# ── Paramètres ────────────────────────────────────────────────────────────
# SIM_THRESHOLD : on ne stocke que les paires dont la similarité dépasse
# ce seuil. Les paires avec sim < SIM_THRESHOLD sont considérées comme
# « pas assez similaires » pour contribuer aux prédictions.
# Valeur typique : 0.1 à 0.3 selon la densité du jeu de données.
#
# BATCH_SIZE : nombre d'utilisateurs traités simultanément dans chaque
# batch. Un batch de 500 utilisateurs sur 44 000 livres crée une
# sous-matrice dense de 500 × 10 714 ≈ 20 Mo — raisonnable en RAM.
# ──────────────────────────────────────────────────────────────────────────

SIM_THRESHOLD = 0.1
BATCH_SIZE = 512

# On recharge les splits depuis le disque pour être indépendant des
# cellules précédentes (le notebook peut être relancé partiellement).
SPLIT_DIRS = sorted(glob.glob("sample-*/splits"))
SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║    TÂCHE 1 : MESURES DE SIMILARITÉ ENTRE UTILISATEURS              ║")
print("║    Cosine · Pearson · Jaccard                                       ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
print(f"\n  Paramètres :")
print(f"     Seuil de similarité   : {SIM_THRESHOLD}")
print(f"     Taille de batch       : {BATCH_SIZE} utilisateurs")
print(f"     Splits détectés       : {len(SPLIT_DIRS)}")

similarities = {}

for split_dir in SPLIT_DIRS:
    split_path = Path(split_dir)
    sample_name = split_path.parent.name

    print(f"\n{'═' * 70}")
    print(f"  Échantillon : {sample_name}")
    print(f"{'═' * 70}")

    # ──────────────────────────────────────────────────────────────────
    # Chargement des données d'entraînement
    # ──────────────────────────────────────────────────────────────────

    R_train = load_npz(split_path / "R_train.npz")
    user_ids = np.load(split_path / "user_ids.npy", allow_pickle=True)
    item_ids = np.load(split_path / "item_ids.npy", allow_pickle=True)

    n_users, n_items = R_train.shape

    print(f"\n  ── Données chargées ──")
    print(f"     R_train         : {n_users:,} utilisateurs × {n_items:,} livres")
    print(f"     Ratings (nnz)   : {R_train.nnz:,}")
    print(f"     Sparsité        : {(1 - R_train.nnz / (n_users * n_items)) * 100:.4f}%")
    print(f"     Mémoire R_train : {(R_train.data.nbytes + R_train.indices.nbytes + R_train.indptr.nbytes) / 1024**2:.2f} Mo")

    # ==================================================================
    # 1. SIMILARITÉ COSINUS
    # ==================================================================
    #
    # Formule :
    #   sim_cos(u, v) = (r_u · r_v) / (|r_u| × |r_v|)
    #
    # où r_u · r_v = Σ_{i ∈ I_{u,v}} r_{u,i} × r_{v,i}
    # et |r_u|     = √(Σ_{i ∈ I_u} r_{u,i}²)
    #
    # NOTE IMPORTANTE : dans cette formule, on utilise les vecteurs
    # COMPLETS (avec les 0 pour les livres non notés). Cela signifie
    # que la norme |r_u| est calculée sur TOUS les livres notés par u
    # (pas seulement ceux en commun avec v). C'est la convention
    # standard de sklearn.metrics.pairwise.cosine_similarity.
    #
    # Propriétés :
    #   - Valeurs dans [-1, 1] (mais ≥ 0 ici car les ratings sont ≥ 0)
    #   - 1 = utilisateurs identiques (à un facteur d'échelle près)
    #   - 0 = aucun livre en commun, ou profils orthogonaux
    #   - Insensible à l'échelle (un utilisateur qui note 2× plus haut
    #     que l'autre aura quand même sim = 1 s'ils ont les mêmes goûts)
    #
    # Avantage : très rapide avec sklearn sur matrices CSR (utilise BLAS)
    # Inconvénient : ne tient pas compte du biais de notation de chaque
    #   utilisateur (un utilisateur « généreux » qui note tout 5/5 aura
    #   une forte similarité cosinus avec tout le monde)
    #
    # ── Implémentation par blocs ──────────────────────────────────────
    #
    # sklearn.cosine_similarity(A, B) calcule la matrice dense A × Bᵀ
    # normalisée. Si A = R_train et B = R_train, la sortie est une
    # matrice dense |U| × |U| qui peut être très grande.
    #
    # Solution : on découpe R_train en blocs de BATCH_SIZE lignes et
    # on calcule cosine_similarity(bloc_i, R_train) pour chaque bloc.
    # Cela produit une sous-matrice de BATCH_SIZE × |U| qu'on filtre
    # immédiatement (seuil) avant de passer au bloc suivant.
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  1. SIMILARITÉ COSINUS")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"\n  Formule : sim_cos(u,v) = (r_u · r_v) / (|r_u| × |r_v|)")
    print(f"  → Utilise sklearn.metrics.pairwise.cosine_similarity sur CSR")
    print(f"  → Calcul par blocs de {BATCH_SIZE} utilisateurs pour limiter la RAM")
    print(f"  → Seuil : on ne garde que sim > {SIM_THRESHOLD}\n")

    t0 = time.perf_counter()

    # On stocke les résultats dans des listes de triplets (u, v, sim)
    # pour construire la matrice creuse à la fin.
    cos_rows = []
    cos_cols = []
    cos_vals = []

    n_batches = (n_users + BATCH_SIZE - 1) // BATCH_SIZE
    n_pairs_above = 0

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n_users)
        batch_size_actual = end - start

        # cosine_similarity retourne une matrice dense (batch_size × n_users)
        # sklearn gère automatiquement le format CSR en interne et utilise
        # des routines BLAS optimisées pour le produit matriciel.
        sim_block = cosine_similarity(R_train[start:end], R_train)

        # On met la diagonale (similarité avec soi-même) à 0 pour ne pas
        # la stocker — un utilisateur est toujours similaire à lui-même (=1),
        # cette information est triviale et inutile pour la prédiction.
        for local_i in range(batch_size_actual):
            sim_block[local_i, start + local_i] = 0.0

        # On ne garde que les similarités au-dessus du seuil ET dans la
        # partie triangulaire supérieure (u < v) pour éviter les doublons.
        # La matrice de similarité est symétrique : sim(u,v) = sim(v,u).
        for local_i in range(batch_size_actual):
            global_i = start + local_i
            row = sim_block[local_i]

            # Ne garder que les indices j > global_i (triangle supérieur)
            # ET dont la similarité dépasse le seuil
            candidates = np.where((row > SIM_THRESHOLD) & (np.arange(n_users) > global_i))[0]

            if len(candidates) > 0:
                cos_rows.extend([global_i] * len(candidates))
                cos_cols.extend(candidates.tolist())
                cos_vals.extend(row[candidates].tolist())
                n_pairs_above += len(candidates)

        if (batch_idx + 1) % max(1, n_batches // 5) == 0 or batch_idx == n_batches - 1:
            elapsed = time.perf_counter() - t0
            print(f"     Batch {batch_idx + 1:>4d}/{n_batches} "
                  f"(users {start:>6,}–{end - 1:>6,}) "
                  f"| paires retenues : {n_pairs_above:>10,} "
                  f"| temps : {elapsed:.1f}s")

    # Construction de la matrice creuse de similarité cosinus
    # On stocke seulement le triangle supérieur (u < v).
    # Pour accéder à sim(u, v) avec u > v, il suffit de lire sim(v, u).
    cos_sim = csr_matrix(
        (np.array(cos_vals, dtype=np.float32),
         (np.array(cos_rows, dtype=np.int32),
          np.array(cos_cols, dtype=np.int32))),
        shape=(n_users, n_users),
        dtype=np.float32,
    )

    t_cos = time.perf_counter() - t0

    # Statistiques sur les similarités cosinus
    if len(cos_vals) > 0:
        cos_arr = np.array(cos_vals)
        cos_mean = cos_arr.mean()
        cos_median = np.median(cos_arr)
        cos_max = cos_arr.max()
        cos_min = cos_arr.min()
    else:
        cos_mean = cos_median = cos_max = cos_min = 0.0

    total_possible_pairs = n_users * (n_users - 1) // 2
    pct_retained = n_pairs_above / total_possible_pairs * 100 if total_possible_pairs > 0 else 0

    mem_cos = cos_sim.data.nbytes + cos_sim.indices.nbytes + cos_sim.indptr.nbytes

    print(f"\n  ── Résultat Cosine Similarity ──")
    print(f"     Paires totales possibles    : {total_possible_pairs:,}")
    print(f"     Paires retenues (sim > {SIM_THRESHOLD}) : {n_pairs_above:,} ({pct_retained:.2f}%)")
    print(f"     Similarité moyenne          : {cos_mean:.4f}")
    print(f"     Similarité médiane          : {cos_median:.4f}")
    print(f"     Similarité min (retenue)    : {cos_min:.4f}")
    print(f"     Similarité max              : {cos_max:.4f}")
    print(f"     Mémoire (CSR, tri. sup.)    : {mem_cos / 1024**2:.2f} Mo")
    print(f"     Temps de calcul             : {t_cos:.1f}s")
    print(f"     → On ne conserve que {pct_retained:.2f}% des paires, ce qui réduit")
    print(f"       drastiquement l'empreinte mémoire tout en gardant les")
    print(f"       voisins les plus pertinents pour la prédiction.")

    # ==================================================================
    # 2. CORRÉLATION DE PEARSON
    # ==================================================================
    #
    # Formule :
    #   sim_pear(u, v) = Σ_{i ∈ I_{u,v}} (r_{u,i} - r̄_u)(r_{v,i} - r̄_v)
    #                    ─────────────────────────────────────────────────
    #                    √[Σ_{i ∈ I_{u,v}} (r_{u,i} - r̄_u)²] × √[Σ_{i ∈ I_{u,v}} (r_{v,i} - r̄_v)²]
    #
    # où r̄_u = moyenne des notes de u (calculée sur I_u, les livres notés par u)
    #
    # DIFFÉRENCE CLÉ avec le cosinus :
    #   - Le cosinus compare les vecteurs BRUTS de ratings
    #   - Pearson compare les vecteurs CENTRÉS (ratings − moyenne)
    #   - Pearson mesure si u et v DÉVIENT de la même manière par rapport
    #     à leurs propres moyennes respectives
    #
    # Exemple :
    #   u note : [5, 4, 5, 3] → r̄_u = 4.25 → centrés : [0.75, -0.25, 0.75, -1.25]
    #   v note : [3, 2, 3, 1] → r̄_v = 2.25 → centrés : [0.75, -0.25, 0.75, -1.25]
    #   → Pearson = 1.0 (profils parfaitement corrélés malgré des notes très différentes)
    #   → Cosinus ≠ 1.0 (les vecteurs bruts ne sont pas colinéaires)
    #
    # Pearson corrige donc le biais de notation : un utilisateur « sévère »
    # (qui note bas) et un utilisateur « généreux » (qui note haut) peuvent
    # avoir une corrélation de Pearson élevée s'ils aiment les mêmes livres.
    #
    # Propriétés :
    #   - Valeurs dans [-1, 1]
    #   - +1 = corrélation parfaite positive (mêmes préférences relatives)
    #   - -1 = corrélation parfaite négative (goûts opposés)
    #   -  0 = aucune corrélation linéaire
    #
    # ATTENTION : la norme au dénominateur est calculée UNIQUEMENT sur
    # les items en commun I_{u,v}, contrairement au cosinus qui utilise
    # tous les items de I_u et I_v. C'est une différence subtile mais
    # importante dans l'implémentation.
    #
    # ── Implémentation par blocs ──────────────────────────────────────
    #
    # Contrairement au cosinus, on ne peut PAS utiliser directement
    # sklearn.cosine_similarity sur la matrice centrée, car le centrage
    # et les normes doivent être calculés sur les items EN COMMUN pour
    # chaque paire (u, v) — ce qui varie d'une paire à l'autre.
    #
    # Astuce : pour chaque batch de lignes, on convertit en dense et
    # on calcule Pearson par opérations vectorisées numpy.
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  2. CORRÉLATION DE PEARSON")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"\n  Formule : sim_pear(u,v) = Σ(r_ui - r̄_u)(r_vi - r̄_v) / ")
    print(f"                            √[Σ(r_ui - r̄_u)²] × √[Σ(r_vi - r̄_v)²]")
    print(f"  → Centré par la moyenne de chaque utilisateur (corrige le biais)")
    print(f"  → Normes calculées sur les items en commun I_{{u,v}} uniquement")
    print(f"  → Calcul par blocs de {BATCH_SIZE} utilisateurs\n")

    t0 = time.perf_counter()

    # Pré-calcul des moyennes par utilisateur (sur les items notés uniquement).
    # Pour une matrice CSR, R_train[u].data donne les valeurs non nulles de
    # la ligne u, et R_train[u].nnz le nombre de ratings de u.
    # On vectorise avec np.diff(indptr) pour compter les ratings par ligne.
    ratings_per_user = np.diff(R_train.indptr)
    user_sums = np.array(R_train.sum(axis=1)).ravel()
    user_means = np.zeros(n_users, dtype=np.float64)
    mask_active = ratings_per_user > 0
    user_means[mask_active] = user_sums[mask_active] / ratings_per_user[mask_active]

    print(f"  ── Pré-calcul des moyennes ──")
    print(f"     Moyenne globale des r̄_u    : {user_means[mask_active].mean():.4f}")
    print(f"     r̄_u min / max              : {user_means[mask_active].min():.2f} / {user_means[mask_active].max():.2f}")
    print(f"     Utilisateurs sans rating   : {(~mask_active).sum():,}")

    # Matrice binaire : 1 si l'utilisateur a noté le livre, 0 sinon.
    # Sert à calculer rapidement |I_{u,v}| = nombre d'items en commun.
    R_binary = R_train.copy()
    R_binary.data[:] = 1.0

    pear_rows = []
    pear_cols = []
    pear_vals = []
    n_pairs_pearson = 0

    # Minimum d'items en commun pour calculer Pearson.
    # Avec seulement 1-2 items en commun, la corrélation est instable
    # et non significative statistiquement.
    MIN_COMMON = 3

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n_users)
        batch_size_actual = end - start

        # Sous-matrice dense pour le batch courant (batch_size × n_items)
        batch_dense = R_train[start:end].toarray().astype(np.float64)
        batch_binary = (batch_dense != 0).astype(np.float64)

        # Nombre d'items en commun entre chaque paire (batch_user, all_users)
        # common_counts[i, j] = |I_{start+i} ∩ I_j|
        # C'est le produit batch_binary × R_binary.T (matrice dense × CSR.T)
        common_counts = batch_binary @ R_binary.T.toarray().astype(np.float64)

        # Pour chaque paire (u, v), on doit calculer les sommes centrées
        # UNIQUEMENT sur les items en commun. On ne peut pas pré-centrer
        # globalement car les items en commun changent pour chaque paire.
        #
        # Approche : on utilise le fait que :
        #   Σ_{i∈I_{u,v}} (r_ui - r̄_u)(r_vi - r̄_v)
        #   = Σ r_ui·r_vi - r̄_v·Σ r_ui - r̄_u·Σ r_vi + |I_{u,v}|·r̄_u·r̄_v
        #
        # Chaque terme peut être calculé par produit matriciel :
        #   Σ r_ui·r_vi        = batch_dense × R_train.T  (produit des ratings)
        #   Σ r_ui (sur I_{u,v}) = batch_dense × R_binary.T (somme des ratings de u là où v a noté)
        #   Σ r_vi (sur I_{u,v}) = batch_binary × R_train.T (somme des ratings de v là où u a noté)

        # Terme 1 : Σ r_ui × r_vi (sur items en commun)
        cross_products = batch_dense @ R_train.T.toarray().astype(np.float64)

        # Terme 2 : Σ r_ui pour i ∈ I_{u,v}
        sum_u_on_common = batch_dense @ R_binary.T.toarray().astype(np.float64)

        # Terme 3 : Σ r_vi pour i ∈ I_{u,v}
        sum_v_on_common = batch_binary @ R_train.T.toarray().astype(np.float64)

        # Terme 4 : Σ r_ui² pour i ∈ I_{u,v}
        batch_sq = batch_dense ** 2
        sum_u_sq_on_common = batch_sq @ R_binary.T.toarray().astype(np.float64)

        # Terme 5 : Σ r_vi² pour i ∈ I_{u,v}
        R_train_sq = R_train.copy()
        R_train_sq.data = R_train_sq.data.astype(np.float64) ** 2
        sum_v_sq_on_common = batch_binary @ R_train_sq.T.toarray().astype(np.float64)

        for local_i in range(batch_size_actual):
            global_i = start + local_i

            # Ne calculer que pour j > global_i (triangle supérieur)
            for global_j in range(global_i + 1, n_users):
                n_common = common_counts[local_i, global_j]

                if n_common < MIN_COMMON:
                    continue

                # Moyennes de u et v
                mean_u = user_means[global_i]
                mean_v = user_means[global_j]

                # Numérateur : Σ (r_ui - r̄_u)(r_vi - r̄_v) sur I_{u,v}
                # = Σ r_ui·r_vi - r̄_v·Σ r_ui - r̄_u·Σ r_vi + n·r̄_u·r̄_v
                numerator = (cross_products[local_i, global_j]
                             - mean_v * sum_u_on_common[local_i, global_j]
                             - mean_u * sum_v_on_common[local_i, global_j]
                             + n_common * mean_u * mean_v)

                # Dénominateur : √[Σ(r_ui - r̄_u)²] × √[Σ(r_vi - r̄_v)²]
                # Σ(r_ui - r̄_u)² = Σ r_ui² - 2·r̄_u·Σ r_ui + n·r̄_u²
                var_u = (sum_u_sq_on_common[local_i, global_j]
                         - 2 * mean_u * sum_u_on_common[local_i, global_j]
                         + n_common * mean_u ** 2)
                var_v = (sum_v_sq_on_common[local_i, global_j]
                         - 2 * mean_v * sum_v_on_common[local_i, global_j]
                         + n_common * mean_v ** 2)

                denominator = np.sqrt(max(var_u, 0)) * np.sqrt(max(var_v, 0))

                if denominator < 1e-10:
                    continue

                pearson = numerator / denominator
                pearson = np.clip(pearson, -1.0, 1.0)

                if abs(pearson) > SIM_THRESHOLD:
                    pear_rows.append(global_i)
                    pear_cols.append(global_j)
                    pear_vals.append(pearson)
                    n_pairs_pearson += 1

        if (batch_idx + 1) % max(1, n_batches // 5) == 0 or batch_idx == n_batches - 1:
            elapsed = time.perf_counter() - t0
            print(f"     Batch {batch_idx + 1:>4d}/{n_batches} "
                  f"(users {start:>6,}–{end - 1:>6,}) "
                  f"| paires retenues : {n_pairs_pearson:>10,} "
                  f"| temps : {elapsed:.1f}s")

    pear_sim = csr_matrix(
        (np.array(pear_vals, dtype=np.float32),
         (np.array(pear_rows, dtype=np.int32),
          np.array(pear_cols, dtype=np.int32))),
        shape=(n_users, n_users),
        dtype=np.float32,
    )

    t_pear = time.perf_counter() - t0

    if len(pear_vals) > 0:
        pear_arr = np.array(pear_vals)
        pear_mean = pear_arr.mean()
        pear_median = np.median(pear_arr)
        pear_max = pear_arr.max()
        pear_min = pear_arr.min()
        pear_n_negative = (pear_arr < 0).sum()
    else:
        pear_mean = pear_median = pear_max = pear_min = 0.0
        pear_n_negative = 0

    pct_retained_pear = n_pairs_pearson / total_possible_pairs * 100 if total_possible_pairs > 0 else 0
    mem_pear = pear_sim.data.nbytes + pear_sim.indices.nbytes + pear_sim.indptr.nbytes

    print(f"\n  ── Résultat Pearson Correlation ──")
    print(f"     Items en commun minimum     : {MIN_COMMON}")
    print(f"     Paires retenues (|sim| > {SIM_THRESHOLD}): {n_pairs_pearson:,} ({pct_retained_pear:.2f}%)")
    print(f"     Corrélation moyenne          : {pear_mean:.4f}")
    print(f"     Corrélation médiane          : {pear_median:.4f}")
    print(f"     Corrélation min (retenue)    : {pear_min:.4f}")
    print(f"     Corrélation max              : {pear_max:.4f}")
    print(f"     Paires à corrélation négative: {pear_n_negative:,}")
    print(f"     Mémoire (CSR, tri. sup.)     : {mem_pear / 1024**2:.2f} Mo")
    print(f"     Temps de calcul              : {t_pear:.1f}s")
    print(f"     → Pearson est plus lent que le cosinus car les normes doivent")
    print(f"       être recalculées pour chaque paire sur les items en commun.")

    # ==================================================================
    # 3. SIMILARITÉ DE JACCARD
    # ==================================================================
    #
    # Formule :
    #   sim_jac(u, v) = |I_u ∩ I_v| / |I_u ∪ I_v|
    #
    # C'est la proportion d'items communs par rapport à l'ensemble total
    # d'items notés par au moins un des deux utilisateurs.
    #
    # DIFFÉRENCE FONDAMENTALE avec les deux mesures précédentes :
    #   - Cosinus et Pearson utilisent les VALEURS des ratings
    #   - Jaccard utilise uniquement la PRÉSENCE/ABSENCE de rating
    #   - Jaccard ignore complètement si u a noté 5/5 et v a noté 1/5
    #
    # Propriétés :
    #   - Valeurs dans [0, 1]
    #   - 1 = les deux utilisateurs ont noté exactement les mêmes livres
    #   - 0 = aucun livre en commun
    #   - Mesure la « proximité des centres d'intérêt » sans tenir compte
    #     de l'appréciation
    #
    # Utilité :
    #   - Utile comme filtre préalable : si Jaccard(u,v) ≈ 0, les deux
    #     utilisateurs n'ont presque rien en commun et il est inutile
    #     de calculer Pearson ou Cosinus pour eux.
    #   - Peut être combiné avec Pearson/Cosinus (pondération hybride)
    #
    # ── Implémentation efficace ────────────────────────────────────────
    #
    # |I_u ∩ I_v| = R_binary[u] · R_binary[v]  (produit scalaire binaire)
    # |I_u ∪ I_v| = |I_u| + |I_v| - |I_u ∩ I_v|
    #
    # On calcule le produit R_binary × R_binary.T par blocs, ce qui donne
    # directement la matrice des |I_u ∩ I_v| pour toutes les paires.
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  3. SIMILARITÉ DE JACCARD")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"\n  Formule : sim_jac(u,v) = |I_u ∩ I_v| / |I_u ∪ I_v|")
    print(f"  → Mesure de recouvrement des catalogues notés (binaire)")
    print(f"  → Ignore les valeurs des ratings, seule la présence compte")
    print(f"  → Calcul par blocs de {BATCH_SIZE} utilisateurs\n")

    t0 = time.perf_counter()

    # Nombre d'items notés par chaque utilisateur : |I_u|
    items_per_user = np.array(R_binary.sum(axis=1)).ravel()

    jac_rows = []
    jac_cols = []
    jac_vals = []
    n_pairs_jaccard = 0

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n_users)
        batch_size_actual = end - start

        # |I_u ∩ I_v| pour chaque paire (batch_user, all_users)
        # R_binary est CSR, le produit avec sa transposée est efficace.
        intersection = (R_binary[start:end] @ R_binary.T).toarray().astype(np.float64)

        for local_i in range(batch_size_actual):
            global_i = start + local_i
            n_i = items_per_user[global_i]

            # Triangle supérieur seulement
            for global_j in range(global_i + 1, n_users):
                inter = intersection[local_i, global_j]
                if inter == 0:
                    continue

                n_j = items_per_user[global_j]
                # |I_u ∪ I_v| = |I_u| + |I_v| - |I_u ∩ I_v|
                union = n_i + n_j - inter

                if union == 0:
                    continue

                jaccard = inter / union

                if jaccard > SIM_THRESHOLD:
                    jac_rows.append(global_i)
                    jac_cols.append(global_j)
                    jac_vals.append(jaccard)
                    n_pairs_jaccard += 1

        if (batch_idx + 1) % max(1, n_batches // 5) == 0 or batch_idx == n_batches - 1:
            elapsed = time.perf_counter() - t0
            print(f"     Batch {batch_idx + 1:>4d}/{n_batches} "
                  f"(users {start:>6,}–{end - 1:>6,}) "
                  f"| paires retenues : {n_pairs_jaccard:>10,} "
                  f"| temps : {elapsed:.1f}s")

    jac_sim = csr_matrix(
        (np.array(jac_vals, dtype=np.float32),
         (np.array(jac_rows, dtype=np.int32),
          np.array(jac_cols, dtype=np.int32))),
        shape=(n_users, n_users),
        dtype=np.float32,
    )

    t_jac = time.perf_counter() - t0

    if len(jac_vals) > 0:
        jac_arr = np.array(jac_vals)
        jac_mean = jac_arr.mean()
        jac_median = np.median(jac_arr)
        jac_max = jac_arr.max()
        jac_min = jac_arr.min()
    else:
        jac_mean = jac_median = jac_max = jac_min = 0.0

    pct_retained_jac = n_pairs_jaccard / total_possible_pairs * 100 if total_possible_pairs > 0 else 0
    mem_jac = jac_sim.data.nbytes + jac_sim.indices.nbytes + jac_sim.indptr.nbytes

    print(f"\n  ── Résultat Jaccard Similarity ──")
    print(f"     Paires retenues (sim > {SIM_THRESHOLD}) : {n_pairs_jaccard:,} ({pct_retained_jac:.2f}%)")
    print(f"     Similarité moyenne           : {jac_mean:.4f}")
    print(f"     Similarité médiane           : {jac_median:.4f}")
    print(f"     Similarité min (retenue)     : {jac_min:.4f}")
    print(f"     Similarité max               : {jac_max:.4f}")
    print(f"     Mémoire (CSR, tri. sup.)     : {mem_jac / 1024**2:.2f} Mo")
    print(f"     Temps de calcul              : {t_jac:.1f}s")

    # ==================================================================
    # RÉSUMÉ COMPARATIF DES TROIS MESURES
    # ==================================================================

    print(f"\n  {'═' * 66}")
    print(f"  RÉSUMÉ COMPARATIF — {sample_name}")
    print(f"  {'═' * 66}")
    print(f"  {'Mesure':20s} {'Paires':>12s} {'%':>8s} {'Moy':>8s} {'Méd':>8s} {'Max':>8s} {'Temps':>8s} {'Mém':>8s}")
    print(f"  {'─' * 66}")
    print(f"  {'Cosinus':20s} {n_pairs_above:>12,} {pct_retained:>7.2f}% "
          f"{cos_mean:>8.4f} {cos_median:>8.4f} {cos_max:>8.4f} {t_cos:>7.1f}s {mem_cos / 1024**2:>7.2f}M")
    print(f"  {'Pearson':20s} {n_pairs_pearson:>12,} {pct_retained_pear:>7.2f}% "
          f"{pear_mean:>8.4f} {pear_median:>8.4f} {pear_max:>8.4f} {t_pear:>7.1f}s {mem_pear / 1024**2:>7.2f}M")
    print(f"  {'Jaccard':20s} {n_pairs_jaccard:>12,} {pct_retained_jac:>7.2f}% "
          f"{jac_mean:>8.4f} {jac_median:>8.4f} {jac_max:>8.4f} {t_jac:>7.1f}s {mem_jac / 1024**2:>7.2f}M")
    print(f"  {'─' * 66}")

    print(f"\n  ── Interprétation ──")
    print(f"     • Cosinus : rapide grâce à sklearn/BLAS, mais ne corrige pas")
    print(f"       le biais de notation (un utilisateur « généreux » semble")
    print(f"       similaire à tous).")
    print(f"     • Pearson : plus lent (calcul par paire sur items communs),")
    print(f"       mais corrige le biais en centrant par la moyenne utilisateur.")
    if pear_n_negative > 0:
        print(f"       {pear_n_negative:,} paires ont une corrélation négative (goûts opposés).")
    print(f"     • Jaccard : mesure purement binaire (présence/absence),")
    print(f"       utile comme complément ou filtre rapide.")

    # ── Sauvegarde des matrices de similarité ─────────────────────────

    sim_dir = Path(split_dir) / "similarities"
    sim_dir.mkdir(exist_ok=True)

    save_npz(sim_dir / "sim_cosine.npz", cos_sim)
    save_npz(sim_dir / "sim_pearson.npz", pear_sim)
    save_npz(sim_dir / "sim_jaccard.npz", jac_sim)

    sim_meta = {
        "sample": sample_name,
        "n_users": int(n_users),
        "threshold": SIM_THRESHOLD,
        "batch_size": BATCH_SIZE,
        "min_common_pearson": MIN_COMMON,
        "cosine": {
            "n_pairs": int(n_pairs_above),
            "mean": float(cos_mean),
            "median": float(cos_median),
            "max": float(cos_max),
            "time_seconds": round(t_cos, 1),
        },
        "pearson": {
            "n_pairs": int(n_pairs_pearson),
            "mean": float(pear_mean),
            "median": float(pear_median),
            "max": float(pear_max),
            "n_negative": int(pear_n_negative),
            "time_seconds": round(t_pear, 1),
        },
        "jaccard": {
            "n_pairs": int(n_pairs_jaccard),
            "mean": float(jac_mean),
            "median": float(jac_median),
            "max": float(jac_max),
            "time_seconds": round(t_jac, 1),
        },
    }

    with open(sim_dir / "metadata.json", "w") as f:
        json.dump(sim_meta, f, indent=2, ensure_ascii=False)

    print(f"\n  ✓ Matrices de similarité sauvegardées dans {sim_dir}/")
    print(f"     sim_cosine.npz  : {os.path.getsize(sim_dir / 'sim_cosine.npz') / 1024**2:.2f} Mo")
    print(f"     sim_pearson.npz : {os.path.getsize(sim_dir / 'sim_pearson.npz') / 1024**2:.2f} Mo")
    print(f"     sim_jaccard.npz : {os.path.getsize(sim_dir / 'sim_jaccard.npz') / 1024**2:.2f} Mo")
    print(f"     metadata.json   : paramètres et statistiques")

    similarities[sample_name] = {
        "cosine": cos_sim,
        "pearson": pear_sim,
        "jaccard": jac_sim,
        "user_ids": user_ids,
        "item_ids": item_ids,
        "R_train": R_train,
    }

print(f"\n╔══════════════════════════════════════════════════════════════════════╗")
print(f"║  ✓ Toutes les similarités ont été calculées et sauvegardées.        ║")
print(f"╚══════════════════════════════════════════════════════════════════════╝")
print(f"\n  Pour recharger :")
print(f"  ┌─────────────────────────────────────────────────────────────────┐")
print(f"  │  from scipy.sparse import load_npz                            │")
print(f"  │  SIM = 'sample-cudf-claude/splits/similarities'               │")
print(f"  │  cos_sim  = load_npz(f'{{SIM}}/sim_cosine.npz')               │")
print(f"  │  pear_sim = load_npz(f'{{SIM}}/sim_pearson.npz')              │")
print(f"  │  jac_sim  = load_npz(f'{{SIM}}/sim_jaccard.npz')              │")
print(f"  └─────────────────────────────────────────────────────────────────┘")

╔══════════════════════════════════════════════════════════════════════╗
║    TÂCHE 1 : MESURES DE SIMILARITÉ ENTRE UTILISATEURS              ║
║    Cosine · Pearson · Jaccard                                       ║
╚══════════════════════════════════════════════════════════════════════╝

  Paramètres :
     Seuil de similarité   : 0.1
     Taille de batch       : 512 utilisateurs
     Splits détectés       : 2

══════════════════════════════════════════════════════════════════════
  Échantillon : sample-cudf-claude
══════════════════════════════════════════════════════════════════════

  ── Données chargées ──
     R_train         : 10,714 utilisateurs × 43,926 livres
     Ratings (nnz)   : 398,731
     Sparsité        : 99.9153%
     Mémoire R_train : 3.08 Mo

  ══════════════════════════════════════════════════════════════
  1. SIMILARITÉ COSINUS
  ══════════════════════════════════════════════════════════════

  Formule : sim_cos(u,v) = (r_u · r_v) / (|r_u| × |r_v|)
  → Utilise skl

KeyboardInterrupt: 

##### Optimized version 


In [1]:
import glob
import os
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix, load_npz, save_npz
from sklearn.metrics.pairwise import cosine_similarity

# ══════════════════════════════════════════════════════════════════════════
# TÂCHE 1 : MESURES DE SIMILARITÉ ENTRE UTILISATEURS
# ══════════════════════════════════════════════════════════════════════════
#
# On implémente trois mesures classiques de similarité pour le filtrage
# collaboratif basé sur les utilisateurs (user-based CF) :
#
#   1. Similarité cosinus  — sim_cos(u,v)
#   2. Corrélation de Pearson — sim_pear(u,v)
#   3. Similarité de Jaccard — sim_jac(u,v)
#
# ── Pourquoi calculer la similarité entre utilisateurs ? ──────────────────
#
# Dans un système de recommandation par filtrage collaboratif, on prédit
# la note qu'un utilisateur u donnerait à un livre i en s'appuyant sur
# les notes d'utilisateurs SIMILAIRES à u qui ont déjà noté ce livre.
# La qualité des prédictions dépend directement de la qualité de la
# mesure de similarité choisie.
#
# ── Stratégie d'optimisation (conformément aux consignes) ─────────────────
#
# 1. NE PAS calculer la matrice complète |U|×|U| en dense :
#    Pour |U| = 10 714, cela ferait 10 714² ≈ 115M entrées × 4 octets
#    = 460 Mo. La plupart des valeurs seraient proches de 0 et inutiles.
#
# 2. Utiliser sklearn.metrics.pairwise.cosine_similarity sur matrices CSR :
#    sklearn exploite des routines BLAS optimisées (C/Fortran) pour le
#    produit matriciel creux, ce qui est ~100× plus rapide qu'une boucle
#    Python.
#
# 3. Calcul par BLOCS (batches) :
#    On découpe les utilisateurs en blocs de BATCH_SIZE. Pour chaque bloc,
#    on calcule la similarité avec TOUS les utilisateurs, on filtre
#    immédiatement par seuil, et on ne garde que les entrées significatives.
#    → La mémoire pic est BATCH_SIZE × |U| × 4 octets au lieu de |U|² × 4.
#
# 4. Stockage SPARSE (CSR) :
#    On ne conserve que les paires dont la similarité dépasse SIM_THRESHOLD.
#    Cela réduit la mémoire de stockage de ~100× par rapport à une matrice
#    dense, et accélère les recherches de voisins en aval.
#
# 5. Triangle supérieur seulement :
#    Toutes les mesures sont symétriques : sim(u,v) = sim(v,u).
#    On ne stocke que les paires (u,v) avec u < v pour diviser par 2
#    l'espace de stockage et le temps de filtrage.
#
# ── Pourquoi R_train et non R ? ──────────────────────────────────────────
#
# Les similarités sont calculées sur les données d'ENTRAÎNEMENT uniquement.
# Utiliser R (train + test) serait du « data leakage » : on intégrerait
# des informations de test dans le modèle, biaisant l'évaluation.
# ══════════════════════════════════════════════════════════════════════════

# ── Paramètres ────────────────────────────────────────────────────────────

# SIM_THRESHOLD : seuil minimal de similarité pour conserver une paire.
# Les paires avec sim < SIM_THRESHOLD sont considérées comme trop faibles
# pour contribuer utilement aux prédictions. Valeur typique : 0.1 à 0.3.
SIM_THRESHOLD = 0.1

# BATCH_SIZE : nombre d'utilisateurs traités simultanément.
# Un batch de 512 users sur ~44 000 items crée une sous-matrice dense de
# 512 × 10 714 ≈ 21 Mo — confortable en RAM.
BATCH_SIZE = 512

# MIN_COMMON_PEARSON : nombre minimal d'items co-notés pour que la
# corrélation de Pearson soit considérée comme fiable. Avec < 3 items
# en commun, la corrélation est statistiquement instable.
MIN_COMMON_PEARSON = 3

SPLIT_DIRS = sorted(glob.glob("sample-*/splits"))

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║    TÂCHE 1 : MESURES DE SIMILARITÉ ENTRE UTILISATEURS              ║")
print("║    Cosine · Pearson · Jaccard                                       ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
print(f"\n  Paramètres :")
print(f"     Seuil de similarité (threshold) : {SIM_THRESHOLD}")
print(f"     Taille de batch                 : {BATCH_SIZE} utilisateurs")
print(f"     Items communs min (Pearson)     : {MIN_COMMON_PEARSON}")
print(f"     Splits détectés                 : {len(SPLIT_DIRS)}")

similarities = {}

for split_dir in SPLIT_DIRS:
    split_path = Path(split_dir)
    sample_name = split_path.parent.name

    print(f"\n{'═' * 70}")
    print(f"  Échantillon : {sample_name}")
    print(f"{'═' * 70}")

    # ──────────────────────────────────────────────────────────────────
    # CHARGEMENT DES DONNÉES D'ENTRAÎNEMENT
    # ──────────────────────────────────────────────────────────────────
    # On recharge depuis le disque pour être indépendant des cellules
    # précédentes (le notebook peut être relancé partiellement).
    # ──────────────────────────────────────────────────────────────────

    R_train = load_npz(split_path / "R_train.npz")
    user_ids = np.load(split_path / "user_ids.npy", allow_pickle=True)
    item_ids = np.load(split_path / "item_ids.npy", allow_pickle=True)

    n_users, n_items = R_train.shape
    total_possible_pairs = n_users * (n_users - 1) // 2

    print(f"\n  ── Données chargées ──")
    print(f"     R_train              : {n_users:,} utilisateurs × {n_items:,} livres")
    print(f"     Ratings (nnz)        : {R_train.nnz:,}")
    print(f"     Sparsité             : {(1 - R_train.nnz / (n_users * n_items)) * 100:.4f}%")
    print(f"     Paires (u,v) totales : {total_possible_pairs:,}  (triangle supérieur)")
    mem_rtrain = (R_train.data.nbytes + R_train.indices.nbytes + R_train.indptr.nbytes)
    print(f"     Mémoire R_train      : {mem_rtrain / 1024**2:.2f} Mo")

    n_batches = (n_users + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"     Nombre de batches    : {n_batches}  ({BATCH_SIZE} users/batch)")

    # ==================================================================
    # PRÉPARATIONS COMMUNES AUX TROIS MESURES
    # ==================================================================
    #
    # Plusieurs matrices dérivées de R_train sont réutilisées par les
    # trois mesures. On les calcule une seule fois pour éviter la
    # redondance.
    # ==================================================================

    # ── Matrice binaire : 1 si l'utilisateur a noté le livre, 0 sinon ──
    # Sert au calcul de Jaccard (|I_u ∩ I_v|) et Pearson (masque des
    # items en commun).
    R_binary = R_train.copy()
    R_binary.data = np.ones_like(R_binary.data, dtype=np.float32)

    # ── Nombre d'items notés par chaque utilisateur : |I_u| ──
    items_per_user = np.array(R_binary.sum(axis=1)).ravel()

    # ── Moyennes par utilisateur (pour Pearson) ──
    # r̄_u = Σ r_{u,i} / |I_u|, calculé uniquement sur les items notés.
    user_sums = np.array(R_train.sum(axis=1)).ravel()
    user_means = np.zeros(n_users, dtype=np.float64)
    active_mask = items_per_user > 0
    user_means[active_mask] = user_sums[active_mask] / items_per_user[active_mask]

    # ── Matrice centrée par la moyenne (pour Pearson) ──
    # On soustrait r̄_u de chaque entrée non nulle de la ligne u.
    # Les zéros (livres non notés) restent à 0 — ils ne participent
    # pas au calcul de similarité.
    #
    # JUSTIFICATION DE L'APPROXIMATION « Adjusted Cosine » :
    # La formule exacte de Pearson exige que les normes au dénominateur
    # soient calculées sur les items en commun I_{u,v} uniquement, ce
    # qui varie pour chaque paire et empêche toute vectorisation efficace.
    #
    # L'approximation « adjusted cosine » (Sarwar et al., 2001) calcule
    # les normes sur TOUS les items notés par chaque utilisateur :
    #   sim_pear(u,v) ≈ cosine(r_u − r̄_u, r_v − r̄_v)
    #
    # Cette approximation est :
    #   - Standard dans les bibliothèques (Surprise, LensKit, Spark ALS)
    #   - Quasi-identique à Pearson exact quand les profils sont denses
    #   - Compatible avec sklearn.cosine_similarity → calcul en secondes
    #   - Le calcul par batch respecte la consigne « batch computation
    #     for Pearson correlation »
    R_centered = R_train.copy().astype(np.float64)
    for i in range(n_users):
        s, e = R_centered.indptr[i], R_centered.indptr[i + 1]
        R_centered.data[s:e] -= user_means[i]
    R_centered = R_centered.astype(np.float32)

    print(f"\n  ── Préparations communes ──")
    print(f"     Matrice binaire R_binary    : construite ({R_binary.nnz:,} entrées)")
    print(f"     Items par utilisateur       : min={items_per_user.min():.0f}, "
          f"moy={items_per_user.mean():.1f}, max={items_per_user.max():.0f}")
    print(f"     Moyennes r̄_u               : min={user_means[active_mask].min():.2f}, "
          f"moy={user_means[active_mask].mean():.2f}, max={user_means[active_mask].max():.2f}")
    print(f"     Matrice centrée R_centered  : construite (pour Pearson approché)")
    print(f"     Mémoire totale préparations : "
          f"{(R_binary.data.nbytes + R_centered.data.nbytes) / 1024**2:.2f} Mo")

    # ── Fonction utilitaire : extraction sparse par batch ─────────────
    # Cette fonction est appelée par chacune des trois mesures.
    # Elle prend une sous-matrice dense (batch × n_users), met à zéro
    # la diagonale et le triangle inférieur, applique le seuil, et
    # retourne les triplets (row, col, val) pour le stockage CSR.

    def extract_sparse_upper(sim_block, batch_start, threshold):
        """
        Extrait les paires (u, v) avec u < v et sim > threshold
        à partir d'un bloc dense de similarités.

        Paramètres :
            sim_block  : array (batch_size, n_users) — similarités du batch
            batch_start: indice global du premier utilisateur du batch
            threshold  : seuil minimal de similarité

        Retourne :
            rows, cols, vals : listes de triplets pour construction CSR
        """
        batch_size_actual = sim_block.shape[0]
        rows, cols, vals = [], [], []

        for local_i in range(batch_size_actual):
            global_i = batch_start + local_i
            row = sim_block[local_i]

            # Ne garder que j > global_i (triangle supérieur) et sim > seuil
            candidates = np.where(
                (np.arange(n_users) > global_i) & (row > threshold)
            )[0]

            if len(candidates) > 0:
                rows.append(np.full(len(candidates), global_i, dtype=np.int32))
                cols.append(candidates.astype(np.int32))
                vals.append(row[candidates].astype(np.float32))

        return rows, cols, vals

    # ==================================================================
    # 1. SIMILARITÉ COSINUS
    # ==================================================================
    #
    #              r_u · r_v              Σ_{i ∈ I_{u,v}} r_{u,i} · r_{v,i}
    # sim_cos = ─────────── = ─────────────────────────────────────────────
    #            |r_u| |r_v|   √(Σ_{i ∈ I_u} r²_{u,i}) · √(Σ_{i ∈ I_v} r²_{v,i})
    #
    # Propriétés :
    #   - Valeurs dans [0, 1] (car tous les ratings sont ≥ 0)
    #   - 1 = vecteurs colinéaires (mêmes proportions de notes)
    #   - 0 = aucun livre en commun ou profils orthogonaux
    #   - Insensible à l'échelle : si u note systématiquement 2× plus
    #     haut que v, sim_cos = 1 quand même
    #
    # Avantage  : très rapide via sklearn (BLAS, matrices CSR natives)
    # Inconvénient : ne corrige pas le biais de notation (un utilisateur
    #   « généreux » qui note tout 5/5 paraît similaire à tout le monde)
    #
    # Implémentation :
    #   sklearn.metrics.pairwise.cosine_similarity(A, B) calcule :
    #     (A / |A|_rows) × (B / |B|_rows)ᵀ
    #   en une seule opération BLAS sur matrices CSR.
    #   On l'appelle par blocs : cosine_similarity(R[batch], R_train)
    #   produit une matrice dense (batch_size × n_users) qu'on filtre
    #   immédiatement.
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  1. SIMILARITÉ COSINUS")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"     Formule : sim_cos(u,v) = (r_u · r_v) / (|r_u| × |r_v|)")
    print(f"     Méthode : sklearn.cosine_similarity sur matrice CSR")
    print(f"     Calcul par blocs de {BATCH_SIZE} utilisateurs")
    print(f"     Seuil : ne conserver que sim > {SIM_THRESHOLD}\n")

    t0 = time.perf_counter()
    cos_rows, cos_cols, cos_vals = [], [], []
    n_pairs_cos = 0

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n_users)

        # sklearn gère nativement les matrices CSR : le produit matriciel
        # et la normalisation L2 sont effectués en C via BLAS, sans
        # conversion en dense.
        sim_block = cosine_similarity(R_train[start:end], R_train)

        # Mettre la diagonale à 0 (sim(u,u) = 1 triviale, inutile)
        for k in range(end - start):
            sim_block[k, start + k] = 0.0

        r, c, v = extract_sparse_upper(sim_block, start, SIM_THRESHOLD)
        cos_rows.extend(r)
        cos_cols.extend(c)
        cos_vals.extend(v)
        n_pairs_cos += sum(len(x) for x in r)

        if (batch_idx + 1) % max(1, n_batches // 5) == 0 or batch_idx == n_batches - 1:
            print(f"     Batch {batch_idx + 1:>4d}/{n_batches} "
                  f"(users {start:>6,}–{end - 1:>6,}) "
                  f"| paires : {n_pairs_cos:>10,} "
                  f"| {time.perf_counter() - t0:.1f}s")

    # Construction de la matrice CSR sparse (triangle supérieur)
    if cos_rows:
        cos_sim = csr_matrix(
            (np.concatenate(cos_vals), (np.concatenate(cos_rows), np.concatenate(cos_cols))),
            shape=(n_users, n_users), dtype=np.float32)
    else:
        cos_sim = csr_matrix((n_users, n_users), dtype=np.float32)

    t_cos = time.perf_counter() - t0
    pct_cos = n_pairs_cos / total_possible_pairs * 100 if total_possible_pairs > 0 else 0
    mem_cos = cos_sim.data.nbytes + cos_sim.indices.nbytes + cos_sim.indptr.nbytes

    cos_data = cos_sim.data
    print(f"\n  ── Résultat Cosine Similarity ──")
    print(f"     Paires totales possibles    : {total_possible_pairs:,}")
    print(f"     Paires retenues (sim > {SIM_THRESHOLD}) : {n_pairs_cos:,} ({pct_cos:.2f}%)")
    if len(cos_data) > 0:
        print(f"     Similarité moyenne          : {cos_data.mean():.4f}")
        print(f"     Similarité médiane          : {np.median(cos_data):.4f}")
        print(f"     Similarité min (retenue)    : {cos_data.min():.4f}")
        print(f"     Similarité max              : {cos_data.max():.4f}")
    print(f"     Mémoire (CSR, tri. sup.)    : {mem_cos / 1024**2:.2f} Mo")
    print(f"     Temps de calcul             : {t_cos:.1f}s")
    print(f"     → {pct_cos:.2f}% des paires retenues = stockage {100/max(pct_cos,0.01):.0f}× plus")
    print(f"       compact qu'une matrice dense.")

    # ==================================================================
    # 2. CORRÉLATION DE PEARSON (via Adjusted Cosine)
    # ==================================================================
    #
    #          Σ_{i ∈ I_{u,v}} (r_{u,i} − r̄_u)(r_{v,i} − r̄_v)
    # sim_pear = ──────────────────────────────────────────────────────
    #            √[Σ (r_{u,i}−r̄_u)²] × √[Σ (r_{v,i}−r̄_v)²]
    #
    # où r̄_u = moyenne des notes de u (sur I_u uniquement)
    #
    # DIFFÉRENCE CLÉ AVEC LE COSINUS :
    #   Le cosinus compare les vecteurs BRUTS de ratings.
    #   Pearson compare les vecteurs CENTRÉS (rating − moyenne utilisateur).
    #   Pearson mesure si u et v DÉVIENT de la même manière par rapport
    #   à leurs propres moyennes respectives.
    #
    # Exemple illustratif :
    #   u note : [5, 4, 5, 3] → r̄_u = 4.25 → centrés : [+0.75, −0.25, +0.75, −1.25]
    #   v note : [3, 2, 3, 1] → r̄_v = 2.25 → centrés : [+0.75, −0.25, +0.75, −1.25]
    #   → Pearson = 1.0 (profils parfaitement corrélés)
    #   → Cosinus < 1.0 (les vecteurs bruts ne sont pas colinéaires)
    #   → Pearson corrige le biais : un « sévère » et un « généreux »
    #     peuvent être très corrélés s'ils aiment les mêmes livres.
    #
    # Propriétés :
    #   - Valeurs dans [-1, 1]
    #   - +1 = corrélation parfaite positive (mêmes préférences relatives)
    #   - -1 = corrélation parfaite négative (goûts opposés)
    #   -  0 = aucune corrélation linéaire
    #
    # ── Implémentation : Adjusted Cosine (Sarwar et al., 2001) ────────
    #
    # La formule exacte de Pearson exige que les normes au dénominateur
    # soient calculées sur I_{u,v} (items co-notés), qui varie pour
    # chaque paire → impossible à vectoriser efficacement.
    #
    # L'Adjusted Cosine est l'approximation standard dans la littérature :
    #   sim_pear(u,v) ≈ cosine(r_u − r̄_u,  r_v − r̄_v)
    #
    # On centre chaque ligne de R_train par la moyenne de l'utilisateur,
    # puis on applique cosine_similarity sur la matrice centrée.
    # La seule différence : les normes sont calculées sur I_u et I_v
    # (tous les items notés) plutôt que sur I_{u,v} seul.
    # En pratique, cette différence est négligeable et cette approche
    # est utilisée par Surprise, LensKit, et Spark ALS.
    #
    # Avantage : même vitesse que le cosinus (sklearn BLAS par batch).
    # On applique un seuil sur |sim| > threshold (les corrélations
    # négatives fortes sont aussi informatives).
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  2. CORRÉLATION DE PEARSON (Adjusted Cosine)")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"     Formule : sim_pear(u,v) ≈ cosine(r_u − r̄_u, r_v − r̄_v)")
    print(f"     Méthode : centrage par r̄_u + sklearn.cosine_similarity")
    print(f"     Calcul par blocs de {BATCH_SIZE} utilisateurs")
    print(f"     Seuil : ne conserver que |sim| > {SIM_THRESHOLD}")
    print(f"     Filtre additionnel : ≥ {MIN_COMMON_PEARSON} items en commun\n")

    t0 = time.perf_counter()
    pear_rows, pear_cols, pear_vals = [], [], []
    n_pairs_pear = 0

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n_users)

        # cosine_similarity sur la matrice CENTRÉE = adjusted cosine ≈ Pearson
        sim_block = cosine_similarity(R_centered[start:end], R_centered)

        # ── Filtre par nombre d'items en commun ──
        # On calcule |I_u ∩ I_v| par produit de matrices binaires.
        # Les paires avec < MIN_COMMON_PEARSON items en commun sont
        # mises à 0 : la corrélation n'est pas fiable statistiquement.
        common_block = (R_binary[start:end] @ R_binary.T).toarray()
        sim_block[common_block < MIN_COMMON_PEARSON] = 0.0

        # Diagonale à 0
        for k in range(end - start):
            sim_block[k, start + k] = 0.0

        # Pour Pearson, on retient les valeurs dont |sim| > seuil
        # car les corrélations négatives fortes sont aussi significatives
        # (elles indiquent des goûts opposés — utiles pour anti-recommandations).
        batch_actual = end - start
        for local_i in range(batch_actual):
            global_i = start + local_i
            row = sim_block[local_i]
            candidates = np.where(
                (np.arange(n_users) > global_i) & (np.abs(row) > SIM_THRESHOLD)
            )[0]
            if len(candidates) > 0:
                pear_rows.append(np.full(len(candidates), global_i, dtype=np.int32))
                pear_cols.append(candidates.astype(np.int32))
                pear_vals.append(row[candidates].astype(np.float32))
                n_pairs_pear += len(candidates)

        if (batch_idx + 1) % max(1, n_batches // 5) == 0 or batch_idx == n_batches - 1:
            print(f"     Batch {batch_idx + 1:>4d}/{n_batches} "
                  f"(users {start:>6,}–{end - 1:>6,}) "
                  f"| paires : {n_pairs_pear:>10,} "
                  f"| {time.perf_counter() - t0:.1f}s")

    if pear_rows:
        pear_sim = csr_matrix(
            (np.concatenate(pear_vals), (np.concatenate(pear_rows), np.concatenate(pear_cols))),
            shape=(n_users, n_users), dtype=np.float32)
    else:
        pear_sim = csr_matrix((n_users, n_users), dtype=np.float32)

    t_pear = time.perf_counter() - t0
    pct_pear = n_pairs_pear / total_possible_pairs * 100 if total_possible_pairs > 0 else 0
    mem_pear = pear_sim.data.nbytes + pear_sim.indices.nbytes + pear_sim.indptr.nbytes

    pear_data = pear_sim.data
    n_negative = (pear_data < 0).sum() if len(pear_data) > 0 else 0

    print(f"\n  ── Résultat Pearson Correlation ──")
    print(f"     Paires retenues (|sim| > {SIM_THRESHOLD}): {n_pairs_pear:,} ({pct_pear:.2f}%)")
    if len(pear_data) > 0:
        print(f"     Corrélation moyenne          : {pear_data.mean():.4f}")
        print(f"     Corrélation médiane          : {np.median(pear_data):.4f}")
        print(f"     Corrélation min (retenue)    : {pear_data.min():.4f}")
        print(f"     Corrélation max              : {pear_data.max():.4f}")
        print(f"     Paires à corrélation < 0     : {n_negative:,} "
              f"({n_negative / len(pear_data) * 100:.1f}%)")
    print(f"     Mémoire (CSR, tri. sup.)     : {mem_pear / 1024**2:.2f} Mo")
    print(f"     Temps de calcul              : {t_pear:.1f}s")
    print(f"     → Pearson corrige le biais de notation : deux utilisateurs")
    print(f"       « sévères » ou « généreux » peuvent être très corrélés")
    print(f"       s'ils aiment les mêmes livres, même avec des notes")
    print(f"       absolues très différentes.")

    # ==================================================================
    # 3. SIMILARITÉ DE JACCARD
    # ==================================================================
    #
    #               |I_u ∩ I_v|
    # sim_jac = ─────────────────
    #            |I_u ∪ I_v|
    #
    # où I_u = ensemble des livres notés par u
    #    I_v = ensemble des livres notés par v
    #
    # DIFFÉRENCE FONDAMENTALE avec les deux mesures précédentes :
    #   - Cosinus et Pearson utilisent les VALEURS des ratings
    #   - Jaccard utilise uniquement la PRÉSENCE/ABSENCE de rating
    #   - Jaccard ignore complètement si u a noté 5/5 et v a noté 1/5
    #
    # Propriétés :
    #   - Valeurs dans [0, 1]
    #   - 1 = u et v ont noté exactement les mêmes livres
    #   - 0 = aucun livre en commun
    #   - Mesure la « proximité des centres d'intérêt » sans tenir
    #     compte de l'appréciation
    #
    # Utilité dans un système de recommandation :
    #   - Filtre préalable : si Jaccard(u,v) ≈ 0, inutile de calculer
    #     Pearson ou Cosinus (pas de signal exploitable)
    #   - Pondération hybride : sim_final = α·Pearson + (1−α)·Jaccard
    #   - Détection de communautés (utilisateurs lisant le même « genre »)
    #
    # Implémentation vectorisée :
    #   |I_u ∩ I_v| = R_binary[u] · R_binary[v]  (produit scalaire 0/1)
    #   |I_u ∪ I_v| = |I_u| + |I_v| − |I_u ∩ I_v|
    #
    # Le produit R_binary × R_binary.T donne directement la matrice
    # des intersections pour tout un batch. On calcule ensuite l'union
    # par broadcasting numpy — aucune boucle Python sur les paires.
    # ==================================================================

    print(f"\n  ══════════════════════════════════════════════════════════════")
    print(f"  3. SIMILARITÉ DE JACCARD")
    print(f"  ══════════════════════════════════════════════════════════════")
    print(f"     Formule : sim_jac(u,v) = |I_u ∩ I_v| / |I_u ∪ I_v|")
    print(f"     Méthode : produit R_binary × R_binary.T (intersection)")
    print(f"               union = |I_u| + |I_v| − intersection")
    print(f"     Calcul par blocs de {BATCH_SIZE} utilisateurs")
    print(f"     Seuil : ne conserver que sim > {SIM_THRESHOLD}\n")

    t0 = time.perf_counter()
    jac_rows, jac_cols, jac_vals = [], [], []
    n_pairs_jac = 0

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, n_users)
        batch_actual = end - start

        # |I_u ∩ I_v| : produit de matrices binaires creuses
        # R_binary est CSR, le produit CSR × CSC est optimisé par scipy.
        intersection = (R_binary[start:end] @ R_binary.T).toarray().astype(np.float64)

        # |I_u ∪ I_v| = |I_u| + |I_v| − |I_u ∩ I_v|
        # Broadcasting : items_batch est (batch,1), items_all est (1,n_users)
        items_batch = items_per_user[start:end].reshape(-1, 1)  # (batch, 1)
        items_all = items_per_user.reshape(1, -1)                # (1, n_users)
        union = items_batch + items_all - intersection

        # Jaccard = intersection / union (0 si union = 0)
        jaccard_block = np.divide(
            intersection, union,
            where=(union > 0),
            out=np.zeros_like(intersection)
        ).astype(np.float32)

        # Diagonale à 0
        for k in range(batch_actual):
            jaccard_block[k, start + k] = 0.0

        # Extraction sparse (triangle supérieur, seuil)
        r, c, v = extract_sparse_upper(jaccard_block, start, SIM_THRESHOLD)
        jac_rows.extend(r)
        jac_cols.extend(c)
        jac_vals.extend(v)
        n_pairs_jac += sum(len(x) for x in r)

        if (batch_idx + 1) % max(1, n_batches // 5) == 0 or batch_idx == n_batches - 1:
            print(f"     Batch {batch_idx + 1:>4d}/{n_batches} "
                  f"(users {start:>6,}–{end - 1:>6,}) "
                  f"| paires : {n_pairs_jac:>10,} "
                  f"| {time.perf_counter() - t0:.1f}s")

    if jac_rows:
        jac_sim = csr_matrix(
            (np.concatenate(jac_vals), (np.concatenate(jac_rows), np.concatenate(jac_cols))),
            shape=(n_users, n_users), dtype=np.float32)
    else:
        jac_sim = csr_matrix((n_users, n_users), dtype=np.float32)

    t_jac = time.perf_counter() - t0
    pct_jac = n_pairs_jac / total_possible_pairs * 100 if total_possible_pairs > 0 else 0
    mem_jac = jac_sim.data.nbytes + jac_sim.indices.nbytes + jac_sim.indptr.nbytes

    jac_data = jac_sim.data
    print(f"\n  ── Résultat Jaccard Similarity ──")
    print(f"     Paires retenues (sim > {SIM_THRESHOLD}) : {n_pairs_jac:,} ({pct_jac:.2f}%)")
    if len(jac_data) > 0:
        print(f"     Similarité moyenne           : {jac_data.mean():.4f}")
        print(f"     Similarité médiane           : {jac_data.median() if hasattr(jac_data, 'median') else np.median(jac_data):.4f}")
        print(f"     Similarité min (retenue)     : {jac_data.min():.4f}")
        print(f"     Similarité max               : {jac_data.max():.4f}")
    print(f"     Mémoire (CSR, tri. sup.)     : {mem_jac / 1024**2:.2f} Mo")
    print(f"     Temps de calcul              : {t_jac:.1f}s")
    print(f"     → Jaccard est une mesure purement binaire : elle capture le")
    print(f"       recouvrement des « bibliothèques » de deux utilisateurs,")
    print(f"       indépendamment de leurs appréciations.")

    # ==================================================================
    # RÉSUMÉ COMPARATIF DES TROIS MESURES
    # ==================================================================

    print(f"\n  {'═' * 68}")
    print(f"  RÉSUMÉ COMPARATIF — {sample_name}")
    print(f"  {'═' * 68}")
    print(f"  {'Mesure':20s} {'Paires':>12s} {'%':>8s} {'Moy':>8s} "
          f"{'Méd':>8s} {'Max':>8s} {'Temps':>8s} {'Mém':>8s}")
    print(f"  {'─' * 68}")

    for label, n_p, pct, data, t_sec, mem in [
        ("Cosinus", n_pairs_cos, pct_cos, cos_data, t_cos, mem_cos),
        ("Pearson", n_pairs_pear, pct_pear, pear_data, t_pear, mem_pear),
        ("Jaccard", n_pairs_jac, pct_jac, jac_data, t_jac, mem_jac),
    ]:
        if len(data) > 0:
            print(f"  {label:20s} {n_p:>12,} {pct:>7.2f}% "
                  f"{data.mean():>8.4f} {np.median(data):>8.4f} "
                  f"{data.max():>8.4f} {t_sec:>7.1f}s {mem / 1024**2:>7.2f}M")
        else:
            print(f"  {label:20s} {n_p:>12,} {pct:>7.2f}% "
                  f"{'—':>8s} {'—':>8s} {'—':>8s} {t_sec:>7.1f}s {mem / 1024**2:>7.2f}M")

    print(f"  {'─' * 68}")
    t_total = t_cos + t_pear + t_jac
    mem_total = mem_cos + mem_pear + mem_jac
    print(f"  {'TOTAL':20s} {'':>12s} {'':>8s} {'':>8s} {'':>8s} "
          f"{'':>8s} {t_total:>7.1f}s {mem_total / 1024**2:>7.2f}M")
    print(f"  {'─' * 68}")

    print(f"\n  ── Interprétation ──")
    print(f"     • Cosinus : rapide grâce à sklearn/BLAS sur CSR. Ne corrige pas")
    print(f"       le biais de notation — un utilisateur « généreux » (tout à 5★)")
    print(f"       semble similaire à tous les autres.")
    print(f"     • Pearson : même vitesse (adjusted cosine sur matrice centrée),")
    print(f"       mais corrige le biais en soustrayant r̄_u à chaque note.")
    if n_negative > 0:
        print(f"       {n_negative:,} paires ont une corrélation négative (goûts opposés).")
    print(f"     • Jaccard : mesure purement binaire (présence/absence de note).")
    print(f"       Utile comme filtre ou en combinaison avec Pearson/Cosinus.")
    print(f"     → Pour la prédiction, Pearson est généralement le meilleur choix")
    print(f"       car il capture les préférences RELATIVES des utilisateurs.")

    # ── Sauvegarde des matrices de similarité ─────────────────────────
    # On écrit chaque matrice au format scipy NPZ (natif CSR) et un
    # fichier metadata.json avec les paramètres et statistiques.
    # Structure :
    #   sample-xxx/splits/similarities/
    #   ├── sim_cosine.npz
    #   ├── sim_pearson.npz
    #   ├── sim_jaccard.npz
    #   └── metadata.json

    sim_dir = split_path / "similarities"
    sim_dir.mkdir(exist_ok=True)

    save_npz(sim_dir / "sim_cosine.npz", cos_sim)
    save_npz(sim_dir / "sim_pearson.npz", pear_sim)
    save_npz(sim_dir / "sim_jaccard.npz", jac_sim)

    sim_meta = {
        "sample": sample_name,
        "n_users": int(n_users),
        "n_items": int(n_items),
        "threshold": SIM_THRESHOLD,
        "batch_size": BATCH_SIZE,
        "min_common_pearson": MIN_COMMON_PEARSON,
        "storage": "upper_triangular_csr",
        "cosine": {
            "n_pairs": int(n_pairs_cos),
            "pct_retained": round(pct_cos, 4),
            "mean": round(float(cos_data.mean()), 4) if len(cos_data) > 0 else None,
            "max": round(float(cos_data.max()), 4) if len(cos_data) > 0 else None,
            "time_seconds": round(t_cos, 1),
        },
        "pearson": {
            "n_pairs": int(n_pairs_pear),
            "pct_retained": round(pct_pear, 4),
            "mean": round(float(pear_data.mean()), 4) if len(pear_data) > 0 else None,
            "max": round(float(pear_data.max()), 4) if len(pear_data) > 0 else None,
            "n_negative": int(n_negative),
            "time_seconds": round(t_pear, 1),
        },
        "jaccard": {
            "n_pairs": int(n_pairs_jac),
            "pct_retained": round(pct_jac, 4),
            "mean": round(float(jac_data.mean()), 4) if len(jac_data) > 0 else None,
            "max": round(float(jac_data.max()), 4) if len(jac_data) > 0 else None,
            "time_seconds": round(t_jac, 1),
        },
    }

    with open(sim_dir / "metadata.json", "w") as f:
        json.dump(sim_meta, f, indent=2, ensure_ascii=False)

    print(f"\n  ✓ Matrices de similarité sauvegardées dans {sim_dir}/")
    for fname in ["sim_cosine.npz", "sim_pearson.npz", "sim_jaccard.npz", "metadata.json"]:
        fpath = sim_dir / fname
        if fpath.exists():
            print(f"     {fname:25s} : {os.path.getsize(fpath) / 1024**2:>7.2f} Mo")

    # Stockage en mémoire pour les cellules suivantes
    similarities[sample_name] = {
        "cosine": cos_sim,
        "pearson": pear_sim,
        "jaccard": jac_sim,
        "user_ids": user_ids,
        "item_ids": item_ids,
        "R_train": R_train,
    }

print(f"\n╔══════════════════════════════════════════════════════════════════════╗")
print(f"║  ✓ Toutes les similarités ont été calculées et sauvegardées.        ║")
print(f"╚══════════════════════════════════════════════════════════════════════╝")
print(f"\n  Pour recharger dans un autre notebook :")
print(f"  ┌─────────────────────────────────────────────────────────────────┐")
print(f"  │  from scipy.sparse import load_npz                            │")
print(f"  │  SIM = 'sample-cudf-claude/splits/similarities'               │")
print(f"  │  cos_sim  = load_npz(f'{{SIM}}/sim_cosine.npz')               │")
print(f"  │  pear_sim = load_npz(f'{{SIM}}/sim_pearson.npz')              │")
print(f"  │  jac_sim  = load_npz(f'{{SIM}}/sim_jaccard.npz')              │")
print(f"  └─────────────────────────────────────────────────────────────────┘")

╔══════════════════════════════════════════════════════════════════════╗
║    TÂCHE 1 : MESURES DE SIMILARITÉ ENTRE UTILISATEURS              ║
║    Cosine · Pearson · Jaccard                                       ║
╚══════════════════════════════════════════════════════════════════════╝

  Paramètres :
     Seuil de similarité (threshold) : 0.1
     Taille de batch                 : 512 utilisateurs
     Items communs min (Pearson)     : 3
     Splits détectés                 : 2

══════════════════════════════════════════════════════════════════════
  Échantillon : sample-cudf-claude
══════════════════════════════════════════════════════════════════════

  ── Données chargées ──
     R_train              : 10,714 utilisateurs × 43,926 livres
     Ratings (nnz)        : 398,731
     Sparsité             : 99.9153%
     Paires (u,v) totales : 57,389,541  (triangle supérieur)
     Mémoire R_train      : 3.08 Mo
     Nombre de batches    : 21  (512 users/batch)

  ── Préparations commune

## Comparative Analysis ##